# 🔥 Flare industrial: LES 3D, radiação térmica e segurança (Colab)

Simulação de um **flare elevado de propano (584 MW)** sob vento cruzado, com os gráficos atualizados **em tempo real** enquanto o CFD resolve, e gravação de um **vídeo MP4** no final.

| Etapa | O que faz |
|---|---|
| 1 | Modelos de referência do **API 521** (fonte pontual) e de **Chamberlain/Shell (1987)**, mais as correlações de comprimento de chama e fração radiante |
| 2 | Termoquímica: equilíbrio (Cantera) em função da fração de mistura Z e tabela com **β-PDF** de submalha |
| 3 | **LES 3D de baixo Mach em PyTorch (GPU)**: Smagorinsky, Z̃ com TVD, Poisson direto em malha esticada, vento com perfil log, radiação no solo com transmissividade atmosférica |
| 4 | Validação (LES × Chamberlain/API) e segurança: zonas do API 521, dose térmica, probits e temperatura de aço |
| 5 | Vídeo final (simulação + resumo) pronto para o LinkedIn |

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Na GPU, o preset `gpu` (Δ = 0,5 m perto da chama, ~0,8 M células) leva poucos minutos; `gpu_fino` (0,35 m) e `gpu_ultra` (0,25 m) refinam a malha. Sem GPU, o preset `cpu` (Δ = 0,7 m) roda em ~30–60 min e encurta um pouco a chama (veja o estudo de malha na seção 6).

A teoria, a álgebra completa e as referências estão no documento de revisão que acompanha este notebook (`docs/revisao_flare_cfd.html`).

In [ ]:
#@title Ambiente: GPU, Cantera e pacote `flarekit`
import subprocess, sys, os
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
print("GPU:", gpu or "nenhuma (vai rodar em CPU, com malha grossa)")
try:
    import cantera
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cantera"], check=False)
os.makedirs("flarekit", exist_ok=True)

In [ ]:
%%writefile flarekit/__init__.py
"""flarekit — simulação de flares industriais: modelos semi-empíricos (API 521, Chamberlain),
termoquímica em fração de mistura, LES 3D de baixo Mach em PyTorch e critérios de segurança."""
__version__ = "0.1.0"

In [ ]:
%%writefile flarekit/props.py
"""Termoquímica do flare: estequiometria, relação de estado em Z e tabela com β-PDF.

A química é tratada no limite de fração de mistura (seção 3.2 e 4.4 do documento):
o estado local depende só de Z (equilíbrio via Cantera, ou Burke–Schumann se o
Cantera não estiver instalado). Na LES, as médias filtradas vêm da integração
na β-PDF com variância de submalha, o que inclui a interação turbulência–química
e a interação turbulência–radiação na emissão.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy.special import betainc

R_U = 8.314462618        # J/(mol K)
SIGMA = 5.670374419e-8   # W/(m2 K4)
P_ATM = 101325.0         # Pa
G = 9.80665              # m/s2

MW = {"O2": 31.998e-3, "N2": 28.014e-3, "CO2": 44.009e-3, "H2O": 18.015e-3}
Y_O2_AIR = MW["O2"] / (MW["O2"] + 3.76 * MW["N2"])
M_AIR = 0.028965


@dataclass(frozen=True)
class Fuel:
    name: str
    nC: int
    nH: int
    M: float          # kg/mol
    LHV: float        # J/kg
    gamma: float      # cp/cv a ~300 K
    soot_yield: float  # kg fuligem / kg combustível (Tewarson, SFPE Handbook)
    cantera_name: str


PROPANE = Fuel("propano", 3, 8, 44.097e-3, 46.35e6, 1.13, 0.024, "C3H8")
METHANE = Fuel("metano", 1, 4, 16.043e-3, 50.03e6, 1.31, 0.001, "CH4")
ETHANE = Fuel("etano", 2, 6, 30.069e-3, 47.51e6, 1.19, 0.013, "C2H6")
FUELS = {f.name: f for f in (PROPANE, METHANE, ETHANE)}


def stoichiometry(fuel: Fuel) -> dict:
    """a = mols de O2 por mol de combustível; s = razão mássica ar/combustível; Z_st."""
    a = fuel.nC + fuel.nH / 4.0
    s = a * (MW["O2"] + 3.76 * MW["N2"]) / fuel.M
    return {"a": a, "s": s, "Z_st": 1.0 / (1.0 + s)}


def burke_schumann(fuel: Fuel, Z: np.ndarray) -> dict:
    """Frações mássicas de combustão completa (química infinitamente rápida)."""
    Z = np.asarray(Z, dtype=float)
    a = fuel.nC + fuel.nH / 4.0
    r_O2 = a * MW["O2"] / fuel.M
    r_CO2 = fuel.nC * MW["CO2"] / fuel.M
    r_H2O = 0.5 * fuel.nH * MW["H2O"] / fuel.M
    burned = np.minimum(Z, (1.0 - Z) * Y_O2_AIR / r_O2)   # kg combustível queimado / kg mistura
    Y = {
        "F": Z - burned,
        "O2": (1.0 - Z) * Y_O2_AIR - burned * r_O2,
        "N2": (1.0 - Z) * (1.0 - Y_O2_AIR),
        "CO2": burned * r_CO2,
        "H2O": burned * r_H2O,
    }
    Y["O2"] = np.maximum(Y["O2"], 0.0)
    Y["burned"] = burned
    return Y


def radcal_ap(species: str, T: np.ndarray) -> np.ndarray:
    """Coeficiente médio de Planck [1/(atm m)] — ajustes RADCAL da TNF Workshop."""
    t = 1000.0 / np.asarray(T, dtype=float)
    if species == "H2O":
        c = (-0.23093, -1.12390, 9.41530, -2.99880, 0.51382, -1.86840e-5)
    elif species == "CO2":
        c = (18.741, -121.310, 273.500, -194.050, 56.310, -5.8169)
    else:
        raise ValueError(species)
    return sum(ci * t**i for i, ci in enumerate(c))


def soot_kappa_coefficient(n: float = 1.57, k: float = 0.56) -> float:
    """κ_P,fuligem = C·f_v·T, com C = 3.83·C0/C2 (dedução da seção 4.6)."""
    C0 = 36 * np.pi * n * k / ((n**2 - k**2 + 2) ** 2 + 4 * n**2 * k**2)
    return 3.83 * C0 / 1.4388e-2


@dataclass
class StateRelation:
    """Estado em função de Z (sem flutuação): T, ρ, emissão, frações."""
    Z: np.ndarray
    T: np.ndarray
    rho: np.ndarray
    M: np.ndarray
    x_H2O: np.ndarray
    x_CO2: np.ndarray
    Y_F: np.ndarray
    Y_soot: np.ndarray
    kappa: np.ndarray
    emission: np.ndarray   # emissão luminosa da fuligem 4σκ_s T⁴ (f_v,max nominal = 1 ppm) [W/m3]
    T_ad_st: float
    source: str


def state_relation(fuel: Fuel, T_fuel: float, T_inf: float, chi_loss: float,
                   n: int = 2001, use_cantera: bool = True, p: float = P_ATM,
                   phi_rich: float = 2.5) -> StateRelation:
    """Relação de estado φ(Z).

    T_eq(Z) vem de equilíbrio HP (Cantera, gri30) ou de Burke–Schumann com c_p efetivo.
    A perda radiativa usa a abordagem de fração radiante constante (como no FDS):
    T(Z) = T_mix(Z) + (1 − χ)·(T_eq(Z) − T_mix(Z)).
    """
    st = stoichiometry(fuel)
    # malha em Z refinada perto de Z_st (onde tudo acontece)
    Zs = st["Z_st"]
    z1 = np.linspace(0.0, 3 * Zs, n // 2, endpoint=False)
    z2 = np.linspace(3 * Zs, 1.0, n - n // 2)
    Z = np.concatenate([z1, z2])
    Y = burke_schumann(fuel, Z)
    T_mix = None
    source = "Burke-Schumann"
    T_eq = None
    M_mix = None
    xH2O = xCO2 = None
    if use_cantera:
        try:
            import cantera as ct
            gas = ct.Solution("gri30.yaml")
            gas.TPX = T_fuel, p, {fuel.cantera_name: 1.0}
            hF, YF = gas.enthalpy_mass, gas.Y.copy()
            gas.TPX = T_inf, p, {"O2": 1.0, "N2": 3.76}
            hA, YA = gas.enthalpy_mass, gas.Y.copy()
            T_eq = np.empty_like(Z); T_mix = np.empty_like(Z); M_mix = np.empty_like(Z)
            xH2O = np.empty_like(Z); xCO2 = np.empty_like(Z)
            iH2O, iCO2 = gas.species_index("H2O"), gas.species_index("CO2")
            # Limite rico de flamabilidade (como o "rich flammability limit" dos códigos
            # comerciais): acima de Z_rich o equilíbrio prevê craqueamento irreal; a mistura
            # é tratada como o estado queimado em Z_rich misturado com combustível frio.
            Z_rich = phi_rich * Zs / (1 - Zs + phi_rich * Zs)
            gas.HPY = Z_rich * hF + (1 - Z_rich) * hA, p, Z_rich * YF + (1 - Z_rich) * YA
            gas.equilibrate("HP")
            h_r, Y_r = gas.enthalpy_mass, gas.Y.copy()
            for i, z in enumerate(Z):
                gas.HPY = z * hF + (1 - z) * hA, p, z * YF + (1 - z) * YA
                T_mix[i] = gas.T
                if z <= Z_rich:
                    if z > 1e-6:
                        gas.equilibrate("HP")
                else:
                    w = (z - Z_rich) / (1 - Z_rich)
                    gas.HPY = (1 - w) * h_r + w * hF, p, (1 - w) * Y_r + w * YF
                T_eq[i] = gas.T; M_mix[i] = gas.mean_molecular_weight / 1000.0
                xH2O[i] = gas.X[iH2O]; xCO2[i] = gas.X[iCO2]
            source = "Cantera (equilíbrio HP, gri30)"
        except Exception:  # noqa: BLE001 — sem Cantera, cai para Burke–Schumann
            T_eq = None
    if T_eq is None:
        cp_mix = 1100.0
        T_mix = (Z * T_fuel * 1700.0 + (1 - Z) * T_inf * 1005.0) / (Z * 1700.0 + (1 - Z) * 1005.0)
        cp_eff = 1420.0   # calibrado para reproduzir T_ad de alcanos leves (seção 3.2)
        T_eq = T_mix + Y["burned"] * fuel.LHV / cp_eff
        del cp_mix
        moles = {k: Y[k] / (fuel.M if k == "F" else MW[k]) for k in ("F", "O2", "N2", "CO2", "H2O")}
        ntot = sum(moles.values())
        M_mix = 1.0 / ntot
        xH2O, xCO2 = moles["H2O"] / ntot, moles["CO2"] / ntot
    T = T_mix + (1.0 - chi_loss) * (T_eq - T_mix)
    rho = p * M_mix / (R_U * T)
    # Fuligem por relação de estado (Sivathanu & Faeth 1990): só no lado rico da chama,
    # nula para φ ≤ 1, máxima em φ = 2, nula de novo em φ = 5. A emissão para os receptores
    # vem dessa zona luminosa; a potência total é escalada depois para X_rad·Q.
    phi = (Z / np.maximum(1 - Z, 1e-12)) / (Zs / (1 - Zs))
    f_shape = np.clip(np.minimum(phi - 1.0, (5.0 - phi) / 3.0), 0.0, 1.0)
    f_v = 1e-6 * f_shape
    Y_soot = f_v * 1800.0 / rho
    kappa_gas = (p / P_ATM) * (xH2O * radcal_ap("H2O", T) + xCO2 * radcal_ap("CO2", T))
    kappa = np.maximum(kappa_gas, 0.0) + soot_kappa_coefficient() * f_v * T
    emission = 4 * SIGMA * soot_kappa_coefficient() * f_v * T * T**4
    T_ad_st = float(np.interp(Zs, Z, T_eq))
    return StateRelation(Z, T, rho, M_mix, xH2O, xCO2, Y["F"], Y_soot, kappa, emission, T_ad_st, source)


@dataclass
class BetaPDFTable:
    """Tabela φ̃(Z̃, s) com s = Z''²/(Z̃(1−Z̃)) ∈ [0, s_max]. Grades uniformes (busca O(1) na GPU)."""
    Zt: np.ndarray
    s: np.ndarray
    rho: np.ndarray
    drho_dZ: np.ndarray
    T: np.ndarray
    emission: np.ndarray
    Y_F: np.ndarray
    Z_st: float
    T_ad_st: float
    rho_inf: float
    source: str


def beta_pdf_table(sr: StateRelation, Z_st: float, nZ: int = 401, ns: int = 17,
                   s_max: float = 0.8) -> BetaPDFTable:
    """Integra φ(Z) na β-PDF de Favre. Usa diferenças da função beta incompleta em
    cada intervalo, que é robusto mesmo quando a < 1 ou b < 1 (PDF singular nas bordas)."""
    Zt = np.linspace(0.0, 1.0, nZ)
    s = np.linspace(0.0, s_max, ns)
    zf = np.linspace(0.0, 1.0, 4001)
    zm = 0.5 * (zf[1:] + zf[:-1])
    inv_rho = 1.0 / np.interp(zm, sr.Z, sr.rho)
    T_m = np.interp(zm, sr.Z, sr.T)
    e_over_rho = np.interp(zm, sr.Z, sr.emission) * inv_rho
    YF_m = np.interp(zm, sr.Z, sr.Y_F)
    out = {k: np.zeros((nZ, ns)) for k in ("inv_rho", "T", "e_over_rho", "Y_F")}
    for j, sj in enumerate(s):
        for i, z in enumerate(Zt):
            var_ok = sj > 1e-6 and 1e-6 < z < 1 - 1e-6
            if not var_ok:
                w = None
            else:
                gam = 1.0 / sj - 1.0
                a, b = z * gam, (1 - z) * gam
                cdf = betainc(a, b, zf)
                w = np.diff(cdf)
                tot = w.sum()
                w = w / tot if tot > 0 else None
            if w is None:
                out["inv_rho"][i, j] = 1.0 / np.interp(z, sr.Z, sr.rho)
                out["T"][i, j] = np.interp(z, sr.Z, sr.T)
                out["e_over_rho"][i, j] = np.interp(z, sr.Z, sr.emission) * out["inv_rho"][i, j]
                out["Y_F"][i, j] = np.interp(z, sr.Z, sr.Y_F)
            else:
                out["inv_rho"][i, j] = w @ inv_rho
                out["T"][i, j] = w @ T_m
                out["e_over_rho"][i, j] = w @ e_over_rho
                out["Y_F"][i, j] = w @ YF_m
    rho = 1.0 / out["inv_rho"]
    drho = np.gradient(rho, Zt, axis=0)
    return BetaPDFTable(Zt, s, rho, drho, out["T"], rho * out["e_over_rho"], out["Y_F"],
                        Z_st, sr.T_ad_st, float(sr.rho[0]), sr.source)

In [ ]:
%%writefile flarekit/semiempirical.py
"""Modelos semi-empíricos de chama e radiação (níveis N0–N1 do documento, seção 3).

Funções vetorizadas em NumPy. Servem de referência de validação para a LES e de
ferramenta de projeto (altura do flare, distâncias de segurança).
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from .props import G, M_AIR, P_ATM, R_U, Fuel, radcal_ap, stoichiometry


# --------------------------------------------------------------------------- tip
def tip_conditions(fuel: Fuel, mdot: float, T_j: float, mach: float, p: float = P_ATM) -> dict:
    """Seção 3.1: gás ideal na pressão ambiente, Mach prescrito."""
    rho_j = p * fuel.M / (R_U * T_j)
    c_j = np.sqrt(fuel.gamma * R_U * T_j / fuel.M)
    u_j = mach * c_j
    d_j = np.sqrt(4 * mdot / (np.pi * rho_j * u_j))
    return {"rho_j": rho_j, "c_j": c_j, "u_j": u_j, "d_j": d_j, "Q": mdot * fuel.LHV}


def air_density(T_inf: float, p: float = P_ATM) -> float:
    return p * M_AIR / (R_U * T_inf)


# ---------------------------------------------------------------- comprimentos
def api_flame_length(Q: float) -> float:
    """Ajuste de Beychok à curva do API 521: L[m] = 0.00323·Q[W]^0.478."""
    return 0.0059 * 0.3048 * (3.412142 * Q) ** 0.478


def delichatsios_length(u_j, d_j, rho_j, rho_inf, Z_st, T_ad, T_inf) -> dict:
    """Seção 3.3: Froude da chama e L* (Delichatsios 1993; Schefer et al. 2006)."""
    Fr = u_j * Z_st**1.5 / ((rho_j / rho_inf) ** 0.25 * np.sqrt((T_ad - T_inf) / T_inf * G * d_j))
    Lstar = 13.5 * Fr**0.4 / (1 + 0.07 * Fr**2) ** 0.2 if Fr < 5 else 23.0
    dstar = d_j * np.sqrt(rho_j / rho_inf)
    return {"Fr_f": Fr, "L_star": Lstar, "d_star": dstar, "L": Lstar * dstar / Z_st}


def heskestad_length(Q: float, D: float) -> float:
    """Altura média de chamas de empuxo (Heskestad): L = 0.235 Q[kW]^(2/5) − 1.02 D."""
    return 0.235 * (Q / 1e3) ** 0.4 - 1.02 * D


def molina_xrad(fuel: Fuel, mdot, L, Z_st, T_ad, rho_f) -> dict:
    """Seção 3.4: X_rad = 9.45e-9 (τ_G a_p T_ad⁴)^0.47, τ_G em ms."""
    n_tot = fuel.nC + fuel.nH / 2 + 3.76 * (fuel.nC + fuel.nH / 4)
    x_CO2, x_H2O = fuel.nC / n_tot, fuel.nH / 2 / n_tot
    a_p = x_CO2 * radcal_ap("CO2", T_ad) + x_H2O * radcal_ap("H2O", T_ad)
    W_f = 0.17 * L
    tau = np.pi / 12 * rho_f * W_f**2 * L * Z_st / mdot * 1000.0
    return {"a_p": float(a_p), "tau_G_ms": float(tau), "X_rad": float(9.45e-9 * (tau * a_p * T_ad**4) ** 0.47)}


# --------------------------------------------------------------- Chamberlain
@dataclass
class Chamberlain:
    D_s: float
    W: float
    Y: float
    L_b0: float
    L_b: float
    R: float
    Ri_Lb0: float
    alpha: float      # graus
    K: float
    b: float
    R_l: float
    W1: float
    W2: float
    A: float
    F_s: float
    SEP: float        # W/m2
    Q: float


def chamberlain(fuel: Fuel, mdot: float, u_j: float, rho_j: float, rho_inf: float,
                u_w: float, theta_jv: float = 90.0) -> Chamberlain:
    """Seção 3.7 (Chamberlain 1987). Ângulos em graus; u_w é o vento na altura do tip."""
    from scipy.optimize import brentq
    D_s = np.sqrt(4 * mdot / (np.pi * rho_inf * u_j))
    M = fuel.M
    W = M / (15.816 * M + 0.0395)
    Ca = 0.024 * (G * D_s / u_j**2) ** (1 / 3)
    Cb, Cc = 0.2, (2.85 / W) ** (2 / 3)
    Y = brentq(lambda y: Ca * y ** (5 / 3) + Cb * y ** (2 / 3) - Cc, 1e-6, 1e6)
    L_b0 = Y * D_s
    L_b = L_b0 * (0.51 * np.exp(-0.4 * u_w) + 0.49) * (1 - 6.07e-3 * (theta_jv - 90))
    Ri = lambda ell: (G / (D_s**2 * u_j**2)) ** (1 / 3) * ell  # noqa: E731
    R = u_w / u_j
    base = (theta_jv - 90) * (1 - np.exp(-25.6 * R))
    if R <= 0.05:
        alpha = base + 8000 * R / Ri(L_b0)
    else:
        alpha = base + (134 + 1726 * np.sqrt(R - 0.026)) / Ri(L_b0)
    K = 0.185 * np.exp(-20 * R) + 0.015
    a_r = np.radians(alpha)
    b = L_b * np.sin(K * a_r) / np.sin(a_r) if abs(alpha) > 1e-9 else K * L_b
    R_l = np.sqrt(L_b**2 - b**2 * np.sin(a_r) ** 2) - b * np.cos(a_r)
    Cp = 1000 * np.exp(-100 * R) + 0.8
    W1 = D_s * (13.5 * np.exp(-6 * R) + 1.5) * (
        1 - (1 - np.sqrt(rho_inf / rho_j) / 15) * np.exp(-70 * Ri(D_s) * Cp * R))
    W2 = L_b * (0.18 * np.exp(-1.5 * R) + 0.31) * (1 - 0.47 * np.exp(-25 * R))
    A = np.pi / 4 * (W1**2 + W2**2) + np.pi / 2 * (W1 + W2) * np.sqrt(R_l**2 + ((W2 - W1) / 2) ** 2)
    F_s = 0.21 * np.exp(-0.00323 * u_j) + 0.11
    Q = mdot * fuel.LHV
    return Chamberlain(D_s, W, Y, L_b0, L_b, R, Ri(L_b0), alpha, K, b, R_l, W1, W2, A, F_s, F_s * Q / A, Q)


def frustum_axis(ch: Chamberlain, tip: np.ndarray, wind_dir: np.ndarray = np.array([1.0, 0, 0])):
    """Centro da base, vetor do eixo (unitário) e base ortonormal do frustum de um tip vertical."""
    a_r = np.radians(ch.alpha)
    wd = wind_dir / np.linalg.norm(wind_dir)
    ax = np.sin(a_r) * wd + np.cos(a_r) * np.array([0, 0, 1.0])
    base = np.asarray(tip, float) + ch.b * np.array([0, 0, 1.0])
    e2 = np.cross(ax, wd); e2 = e2 / np.linalg.norm(e2) if np.linalg.norm(e2) > 1e-9 else np.array([0, 1.0, 0])
    e1 = np.cross(e2, ax)
    return base, ax, e1, e2


def frustum_panels(ch: Chamberlain, tip, n_ax: int = 48, n_th: int = 64, n_r: int = 12):
    """Painéis (centro, normal, área) da superfície lateral e dos dois discos."""
    base, ax, e1, e2 = frustum_axis(ch, tip)
    s = (np.arange(n_ax) + 0.5) / n_ax
    th = (np.arange(n_th) + 0.5) / n_th * 2 * np.pi
    S, TH = np.meshgrid(s, th, indexing="ij")
    r = (ch.W1 + (ch.W2 - ch.W1) * S) / 2
    slope = (ch.W2 - ch.W1) / 2 / ch.R_l
    radial = np.cos(TH)[..., None] * e1 + np.sin(TH)[..., None] * e2
    pts = base + (S[..., None] * ch.R_l) * ax + r[..., None] * radial
    nrm = radial - slope * ax
    nrm /= np.linalg.norm(nrm, axis=-1, keepdims=True)
    dA = r * (2 * np.pi / n_th) * (ch.R_l / n_ax) * np.sqrt(1 + slope**2)
    P, N, A = [pts.reshape(-1, 3)], [nrm.reshape(-1, 3)], [dA.ravel()]
    for frac, rad, sgn in ((0.0, ch.W1 / 2, -1.0), (1.0, ch.W2 / 2, 1.0)):
        rr = (np.arange(n_r) + 0.5) / n_r * rad
        RR, TT = np.meshgrid(rr, th, indexing="ij")
        c = base + frac * ch.R_l * ax
        p = c + RR[..., None] * (np.cos(TT)[..., None] * e1 + np.sin(TT)[..., None] * e2)
        P.append(p.reshape(-1, 3)); N.append(np.tile(sgn * ax, (p.size // 3, 1)))
        A.append((RR * (rad / n_r) * (2 * np.pi / n_th)).ravel())
    return np.vstack(P), np.vstack(N), np.concatenate(A)


def flame_center_chamberlain(ch: Chamberlain, tip) -> np.ndarray:
    base, ax, *_ = frustum_axis(ch, tip)
    return base + 0.5 * ch.R_l * ax


# --------------------------------------------------------- transmissividade
def p_sat_water(T: float) -> float:
    """Pa, Antoine (TNO Yellow Book)."""
    return float(np.exp(23.18986 - 3816.42 / (T - 46.13)))


def tau_bagster(X, T_inf: float, RH: float):
    X = np.maximum(np.asarray(X, float), 1e-3)
    return np.minimum(1.0, 2.02 * (RH * p_sat_water(T_inf) * X) ** -0.09)


def tau_wayne(X, T_inf: float, RH: float, co2_ppm: float = 335.0):
    """Wayne (1991), como implementado no HyRAM+."""
    X = np.maximum(np.asarray(X, float), 1e-2)
    psat_mmHg = np.exp(20.386 - 5132.0 / T_inf)
    XH2O = RH * X * psat_mmHg * 2.88651e2 / T_inf
    XCO2 = X * 273.0 / T_inf * co2_ppm / 335.0
    lw, lc = np.log10(XH2O), np.log10(XCO2)
    return np.clip(1.006 - 0.01171 * lw - 0.02368 * lw**2 - 0.03188 * lc + 0.001164 * lc**2, 0.0, 1.0)


# --------------------------------------------------- fluxo nos receptores
def view_vector(receivers: np.ndarray, P: np.ndarray, N: np.ndarray, dA: np.ndarray,
                chunk: int = 512):
    """Seção 3.8: V = Σ cosβ_f dA/(π r²) ê (só a parte visível, cosβ_f > 0). F = n·V; F_max = |V|.
    Retorna V (n_rec, 3) e a distância média ponderada até a chama (para τ)."""
    receivers = np.atleast_2d(receivers)
    V = np.zeros_like(receivers, dtype=float)
    Xm = np.zeros(len(receivers))
    for i0 in range(0, len(receivers), chunk):
        r = receivers[i0:i0 + chunk]
        d = P[None, :, :] - r[:, None, :]
        dist = np.linalg.norm(d, axis=-1)
        e = d / dist[..., None]
        cos_f = np.maximum(-(e * N[None]).sum(-1), 0.0)
        w = cos_f * dA[None] / (np.pi * dist**2)
        V[i0:i0 + chunk] = (w[..., None] * e).sum(1)
        Xm[i0:i0 + chunk] = (w * dist).sum(1) / np.maximum(w.sum(1), 1e-30)
    return V, Xm


def q_chamberlain(receivers, ch: Chamberlain, tip, T_inf: float, RH: float,
                  normal: np.ndarray | None = None) -> np.ndarray:
    """Fluxo incidente [W/m2] pelo modelo de chama sólida; normal=None → pior orientação."""
    P, N, dA = frustum_panels(ch, tip)
    V, Xm = view_vector(np.asarray(receivers, float), P, N, dA)
    F = np.linalg.norm(V, axis=1) if normal is None else np.maximum(V @ np.asarray(normal, float), 0.0)
    return ch.SEP * F * tau_wayne(np.maximum(Xm - (ch.W1 + ch.W2) / 4, 1.0), T_inf, RH)


def q_point_source(receivers, center, F_rad: float, Q: float, T_inf: float, RH: float) -> np.ndarray:
    """Seção 3.5 (Hajek–Ludwig/API 521): q = τ F Q / (4π D²), pior orientação."""
    D = np.linalg.norm(np.atleast_2d(receivers) - np.asarray(center, float)[None], axis=1)
    return tau_wayne(D, T_inf, RH) * F_rad * Q / (4 * np.pi * D**2)


@dataclass
class Scenario:
    """Cenário do flare e todas as previsões semi-empíricas que servem de referência."""
    fuel: Fuel
    mdot: float = 12.6
    T_j: float = 311.0
    mach: float = 0.5
    H: float = 30.0
    u_w: float = 8.9
    T_inf: float = 298.15
    RH: float = 0.5

    def __post_init__(self):
        self.tip = tip_conditions(self.fuel, self.mdot, self.T_j, self.mach)
        self.rho_inf = air_density(self.T_inf)
        self.st = stoichiometry(self.fuel)
        self.Q = self.tip["Q"]
        self.L_api = api_flame_length(self.Q)
        self.cham = chamberlain(self.fuel, self.mdot, self.tip["u_j"], self.tip["rho_j"],
                                self.rho_inf, self.u_w)
        self.cham0 = chamberlain(self.fuel, self.mdot, self.tip["u_j"], self.tip["rho_j"],
                                 self.rho_inf, 0.0)
        self.tip_xyz = np.array([0.0, 0.0, self.H])

    def references(self, T_ad: float, rho_f: float | None = None) -> dict:
        """Correlações de referência. ρ_f padrão: produtos estequiométricos a T_ad (gás ideal)."""
        if rho_f is None:
            f = self.fuel
            a = f.nC + f.nH / 4
            n = f.nC + f.nH / 2 + 3.76 * a
            M_p = (f.nC * 44.009e-3 + f.nH / 2 * 18.015e-3 + 3.76 * a * 28.014e-3) / n
            rho_f = P_ATM * M_p / (R_U * T_ad)
        de = delichatsios_length(self.tip["u_j"], self.tip["d_j"], self.tip["rho_j"], self.rho_inf,
                                 self.st["Z_st"], T_ad, self.T_inf)
        mo = molina_xrad(self.fuel, self.mdot, de["L"], self.st["Z_st"], T_ad, rho_f)
        return {"delichatsios": de, "molina": mo, "L_api": self.L_api,
                "L_heskestad": heskestad_length(self.Q, self.tip["d_j"])}

In [ ]:
%%writefile flarekit/les.py
"""LES 3D de baixo Mach para um flare elevado em vento cruzado (PyTorch; GPU se houver).

Formulação (seções 4.1–4.6 do documento), no espírito do FDS:
  * baixo Mach, densidade variável: ρ = ρ(Z̃, s) da tabela β-PDF (química rápida + equilíbrio);
  * restrição de divergência  ∇·u = S_m/ρ − (1/ρ²)(dρ/dZ)[∇·(ρD_t∇Z) + S_m(1−Z)];
  * momento em forma advectiva (upwind de 3ª ordem), H = p̃/ρ, termo baroclínico defasado
    (H∇ρ/ρ), empuxo, Smagorinsky;
  * Poisson de coeficiente constante ∇²H = (∇·u* − D)/Δt, resolvido de forma direta pela
    diagonalização dos operadores 1D (autoproblema generalizado B v = λ M v em malha esticada);
  * fração de mistura com advecção TVD (Superbee), variância de submalha Z''² = C_Z Δ² |∇Z̃|²;
  * fonte de combustível volumétrica no tip com o fluxo de quantidade de movimento do jato real;
  * vento com perfil log e tensão de parede no solo; RK2 (SSP) com projeção em cada estágio.

Malha deslocada (MAC) e não uniforme por produto tensorial: fina (Δ_f) numa caixa em volta do
tip e da chama, crescendo geometricamente até Δ_max fora dela.
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass

import numpy as np
import scipy.linalg
import torch

from .props import G, BetaPDFTable

KAPPA = 0.41


# ----------------------------------------------------------------- utilidades
def _sl(a, axis, start, stop=None):
    idx = [slice(None)] * a.ndim
    idx[axis] = slice(start, stop)
    return a[tuple(idx)]


def _pad(a, axis, n=1):
    lo, hi = _sl(a, axis, 0, 1), _sl(a, axis, -1, None)
    return torch.cat([lo] * n + [a] + [hi] * n, dim=axis)


def _avg(a, axis):
    return 0.5 * (_sl(a, axis, 1, None) + _sl(a, axis, 0, -1))


def _to_faces(c, axis):
    """Centro → faces ao longo de axis (faces de borda recebem o valor da célula de borda)."""
    return _avg(_pad(c, axis), axis)


def _diff(a, axis):
    return _sl(a, axis, 1, None) - _sl(a, axis, 0, -1)


def _bshape(arr, axis, ndim=3):
    shape = [1] * ndim
    shape[axis] = -1
    return arr.reshape(shape)


def stretched_faces(lo, hi, f_lo, f_hi, d_fine, ratio=1.1, d_max=2.5):
    """Faces com Δ = d_fine em [f_lo, f_hi] e crescimento geométrico (ratio) até d_max fora."""
    nf = int(round((f_hi - f_lo) / d_fine))
    fine = f_lo + d_fine * np.arange(nf + 1)

    def grow(start, end, sign):
        out, x, d = [], start, d_fine
        while sign * (end - x) > 1e-9:
            d = min(d * ratio, d_max)
            x = x + sign * d
            out.append(x)
        if out:
            out[-1] = end
            if len(out) > 1 and abs(out[-1] - out[-2]) < 0.5 * d:
                out.pop(-2)
        return out

    up = grow(fine[-1], hi, +1)
    down = grow(fine[0], lo, -1)
    return np.concatenate([np.array(down[::-1]), fine, np.array(up)])


@dataclass
class LESConfig:
    # domínio e malha
    x_range: tuple = (-40.0, 130.0)
    y_range: tuple = (-40.0, 40.0)
    z_top: float = 100.0
    d_fine: float = 0.35
    fine_x: tuple = (-4.0, 30.0)
    fine_y: tuple = (-6.0, 6.0)
    fine_z: tuple = (-2.0, 24.0)       # relativo à altura do tip
    ratio: float = 1.1
    d_max: float = 2.5
    # cenário
    H: float = 30.0            # altura do tip [m]
    u_ref: float = 8.9         # vento na altura do tip [m/s]
    z0: float = 0.03           # rugosidade aerodinâmica [m]
    mdot: float = 12.6         # vazão de combustível [kg/s]
    u_jet: float = 128.7       # velocidade real no tip (fluxo de quantidade de movimento)
    src_cells: int = 2         # lado da fonte, em células finas
    X_rad: float = 0.25        # fração radiante usada para escalar a emissão
    T_inf: float = 298.15
    RH: float = 0.5
    # modelos e numérica
    sgs: str = "smagorinsky"   # "smagorinsky" ou "wale"
    Cs: float = 0.10           # valor usual em escoamentos cisalhantes livres
    Cw: float = 0.5
    Sc_t: float = 0.7
    C_Z: float = 0.15
    cfl: float = 0.5
    dt_max: float = 0.05
    upwind: float = 0.0        # fração de upwind de 1ª ordem (0 = upwind de 3ª ordem puro)
    limiter: str = "superbee"  # limitador TVD de Z (Superbee, como no FDS); "mc" ou "vanleer"
    inflow_noise: float = 0.08
    jet_noise: float = 0.0     # impulsos aleatórios na fonte (desligado por padrão)
    baroclinic: bool = True
    # radiação no solo
    receiver_dx: float = 3.0
    receiver_z: float = 1.5
    emitter_bin: float = 1.5   # agrupa emissores em blocos deste tamanho [m]
    device: str = "auto"
    seed: int = 0

    @staticmethod
    def preset(name: str, **kw) -> "LESConfig":
        presets = {
            # GPU (Colab T4): Δ = 0,5 m perto da chama (~0,8 M células). Validado: L e inclinação
            # sob vento de 8,9 m/s a +2% e +10% de Chamberlain (1987)
            "gpu": dict(d_fine=0.5, d_max=3.0),
            # GPU, malha fina: Δ = 0,35 m (~1,8 M células)
            "gpu_fino": dict(d_fine=0.35),
            # GPU grande (A100/L4): Δ = 0,25 m (~4 M células)
            "gpu_ultra": dict(d_fine=0.25, ratio=1.08),
            # CPU: Δ = 0,7 m num domínio menor
            "cpu": dict(d_fine=0.7, d_max=3.5, x_range=(-30.0, 100.0), y_range=(-32.0, 32.0), z_top=90.0,
                        receiver_dx=4.0, emitter_bin=2.0),
            # teste rápido (malha grossa: só para verificar o fluxo do notebook)
            "teste": dict(d_fine=1.5, d_max=3.0, x_range=(-24.0, 60.0), y_range=(-21.0, 21.0),
                          z_top=75.0, receiver_dx=6.0, emitter_bin=3.0),
        }
        cfg = presets[name].copy()
        if kw.get("u_ref", 1.0) <= 0.0 and name != "teste":
            # sem vento: chama vertical e mais longa → caixa fina centrada e mais alta
            cfg.update(x_range=(-50.0, 50.0), y_range=(-50.0, 50.0), z_top=110.0,
                       fine_x=(-7.0, 7.0), fine_y=(-7.0, 7.0), fine_z=(-2.0, 56.0))
        cfg.update(kw)
        return LESConfig(**cfg)


class FlareLES:
    def __init__(self, cfg: LESConfig, table: BetaPDFTable, Q: float):
        self.cfg = cfg
        self.Q = Q   # potência do flare (ṁ·PCI) [W], normaliza a radiação
        dev = cfg.device
        if dev == "auto":
            dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.dev = torch.device(dev)
        self.ft = torch.float32
        torch.manual_seed(cfg.seed)
        t = lambda a: torch.as_tensor(np.asarray(a, dtype=np.float64), dtype=self.ft, device=self.dev)  # noqa: E731
        self._t = t
        df = cfg.d_fine

        # ---- malha (x = 0, y = 0 e z = H são faces)
        snap = lambda a: round(a / df) * df  # noqa: E731
        H = snap(cfg.H)
        self.xf = stretched_faces(cfg.x_range[0], cfg.x_range[1], snap(cfg.fine_x[0]), snap(cfg.fine_x[1]),
                                  df, cfg.ratio, cfg.d_max)
        self.yf = stretched_faces(cfg.y_range[0], cfg.y_range[1], snap(cfg.fine_y[0]), snap(cfg.fine_y[1]),
                                  df, cfg.ratio, cfg.d_max)
        self.zf = stretched_faces(0.0, cfg.z_top, H + snap(cfg.fine_z[0]), H + snap(cfg.fine_z[1]),
                                  df, cfg.ratio, cfg.d_max)
        self.nx, self.ny, self.nz = len(self.xf) - 1, len(self.yf) - 1, len(self.zf) - 1
        self.n_cells = self.nx * self.ny * self.nz
        self.xc, self.yc, self.zc = [0.5 * (f[1:] + f[:-1]) for f in (self.xf, self.yf, self.zf)]
        self.wind = cfg.u_ref > 0.0
        # espaçamentos: largura das células (n), distância centro-centro nas faces (n+1)
        # e largura do volume de controle das faces (n+1)
        self.dc, self.df_, self.dcf = [], [], []
        for f, c in ((self.xf, self.xc), (self.yf, self.yc), (self.zf, self.zc)):
            dc = np.diff(f)
            self.dc.append(dc)
            self.df_.append(np.concatenate([[dc[0]], np.diff(c), [dc[-1]]]))
            self.dcf.append(np.concatenate([[dc[0]], 0.5 * (dc[1:] + dc[:-1]), [dc[-1]]]))
        self.Dc = [_bshape(t(a), ax) for ax, a in enumerate(self.dc)]
        self.Df = [_bshape(t(a), ax) for ax, a in enumerate(self.df_)]
        self.Dcf = [_bshape(t(a), ax) for ax, a in enumerate(self.dcf)]
        vol = np.einsum("i,j,k->ijk", *self.dc)
        self.vol = t(vol)
        self.delta = self.vol ** (1.0 / 3.0)
        self.inv_d2 = 1 / self.Dc[0] ** 2 + 1 / self.Dc[1] ** 2 + 1 / self.Dc[2] ** 2

        # ---- tabela termoquímica
        self.tab = table
        self.tab_rho = t(table.rho); self.tab_drho = t(table.drho_dZ); self.tab_T = t(table.T)
        self.tab_e = t(table.emission)
        self.nZt, self.ns = table.rho.shape
        self.s_max = float(table.s[-1])
        self.rho_inf = float(table.rho_inf)
        self.Z_st = float(table.Z_st)

        # ---- campos
        nx, ny, nz = self.nx, self.ny, self.nz
        z = lambda *s: torch.zeros(*s, dtype=self.ft, device=self.dev)  # noqa: E731
        self.u, self.v, self.w = z(nx + 1, ny, nz), z(nx, ny + 1, nz), z(nx, ny, nz + 1)
        self.Z, self.Hp = z(nx, ny, nz), z(nx, ny, nz)

        # ---- vento (perfil log ajustado para u_ref na altura do tip)
        self.u_star = KAPPA * cfg.u_ref / math.log((H + cfg.z0) / cfg.z0) if self.wind else 0.0
        U_prof = self.u_star / KAPPA * np.log((self.zc + cfg.z0) / cfg.z0)
        self.U_in = t(U_prof).view(1, 1, nz).expand(1, ny, nz).clone()
        if self.wind:
            self.u[:] = self.U_in
        self.Cd_wall = (KAPPA / math.log((0.5 * self.dc[2][0]) / cfg.z0)) ** 2
        self.noise = z(1, ny, nz)

        # ---- fonte de combustível (bloco src×src×1 logo acima do tip)
        i0 = int(np.argmin(np.abs(self.xf))); j0 = int(np.argmin(np.abs(self.yf)))
        n2 = cfg.src_cells // 2
        self.kH = int(np.argmin(np.abs(self.zf - H)))
        self.isrc = slice(i0 - n2, i0 - n2 + cfg.src_cells)
        self.jsrc = slice(j0 - n2, j0 - n2 + cfg.src_cells)
        Sm = z(nx, ny, nz)
        V_src = float(vol[self.isrc, self.jsrc, self.kH].sum())
        Sm[self.isrc, self.jsrc, self.kH] = cfg.mdot / V_src
        self.Sm = Sm
        self.Sm_u, self.Sm_v, self.Sm_w = _to_faces(Sm, 0), _to_faces(Sm, 1), _to_faces(Sm, 2)
        self.src_u, self.src_v = self.Sm_u.gt(0).to(self.ft), self.Sm_v.gt(0).to(self.ft)
        self.tip = np.array([0.0, 0.0, self.zf[self.kH]])
        self.src_area = float(self.dc[0][self.isrc].sum() * self.dc[1][self.jsrc].sum())

        # ---- Poisson (diagonalização 1D, autoproblema generalizado)
        bx = ("N" if self.wind else "D", "D")
        self.bc = (bx, ("D", "D"), ("N", "D"))
        self._poisson_setup()

        # ---- receptores no solo
        rx = np.arange(self.xf[0] + cfg.receiver_dx / 2, self.xf[-1], cfg.receiver_dx)
        ry = np.arange(self.yf[0] + cfg.receiver_dx / 2, self.yf[-1], cfg.receiver_dx)
        self.rec_x, self.rec_y = rx, ry
        RX, RY = np.meshgrid(rx, ry, indexing="ij")
        self.receivers = t(np.stack([RX, RY, np.full_like(RX, cfg.receiver_z)], -1).reshape(-1, 3))

        # ---- coordenadas das células
        Xc, Yc, Zc = np.meshgrid(self.xc, self.yc, self.zc, indexing="ij")
        self.cell_xyz = t(np.stack([Xc, Yc, Zc], -1)).reshape(-1, 3)
        self.dist_tip = torch.linalg.norm(self.cell_xyz - t(self.tip), dim=-1).reshape(nx, ny, nz)
        self.j_mid = int(np.argmin(np.abs(self.yc)))

        # ---- estado do tempo e médias
        self.time = 0.0; self.step_n = 0; self.dt = cfg.dt_max; self.wall = 0.0
        self._numax = 0.0; self._last_cfl = 0.0
        self.history = {k: [] for k in ("t", "L", "tilt", "qmax", "Tmax", "dt", "cfl")}
        self.avg_n = 0.0
        self.avg_T, self.avg_I = z(nx, ny, nz), z(nx, ny, nz)
        self.avg_q = z(len(rx) * len(ry))
        self.q_inst = torch.zeros_like(self.avg_q)
        self._update_props(self.Z)

    # ---------------------------------------------------------------- Poisson
    def _lap1d(self, axis):
        dc, dfa = self.dc[axis], self.df_[axis]
        lo, hi = self.bc[axis]
        n = len(dc)
        # A = M⁻¹B com M = diag(Δc) e B simétrica: B[i,i±1] = 1/Δf
        B = np.zeros((n, n))
        off = 1.0 / dfa[1:-1]
        idx = np.arange(n - 1)
        B[idx, idx + 1] = off
        B[idx + 1, idx] = off
        diag = np.zeros(n)
        diag[:-1] -= off
        diag[1:] -= off
        if lo == "D":
            diag[0] -= 2.0 / dc[0]
        if hi == "D":
            diag[-1] -= 2.0 / dc[-1]
        B[np.arange(n), np.arange(n)] = diag
        lam, V = scipy.linalg.eigh(B, np.diag(dc))      # Vᵀ M V = I  →  V⁻¹ = Vᵀ M
        return self._t(lam), self._t(V), self._t(V.T * dc[None, :])

    def _poisson_setup(self):
        (lx, Vx, Wx), (ly, Vy, Wy), (lz, Vz, Wz) = (self._lap1d(0), self._lap1d(1), self._lap1d(2))
        self.V = (Vx, Vy, Vz)
        self.Vi = (Wx, Wy, Wz)
        self.lam = lx.view(-1, 1, 1) + ly.view(1, -1, 1) + lz.view(1, 1, -1)

    def poisson(self, rhs):
        Wx, Wy, Wz = self.Vi
        Vx, Vy, Vz = self.V
        r = torch.einsum("ai,ijk->ajk", Wx, rhs)
        r = torch.einsum("bj,ajk->abk", Wy, r)
        r = torch.einsum("ck,abk->abc", Wz, r)
        r = r / self.lam
        r = torch.einsum("ia,abc->ibc", Vx, r)
        r = torch.einsum("jb,ibc->ijc", Vy, r)
        return torch.einsum("kc,ijc->ijk", Vz, r)

    def _grad_face(self, Hc, axis):
        lo, hi = self.bc[axis]
        g_lo = _sl(Hc, axis, 0, 1) * (1.0 if lo == "N" else -1.0)
        g_hi = _sl(Hc, axis, -1, None) * (1.0 if hi == "N" else -1.0)
        return _diff(torch.cat([g_lo, Hc, g_hi], dim=axis), axis) / self.Df[axis]

    def divergence(self, u, v, w):
        return _diff(u, 0) / self.Dc[0] + _diff(v, 1) / self.Dc[1] + _diff(w, 2) / self.Dc[2]

    def project(self, u, v, w, D, coef):
        rhs = (self.divergence(u, v, w) - D) / coef
        Hc = self.poisson(rhs)
        return (u - coef * self._grad_face(Hc, 0), v - coef * self._grad_face(Hc, 1),
                w - coef * self._grad_face(Hc, 2), Hc)

    # ------------------------------------------------------------ propriedades
    def _lookup(self, tab, Z, s):
        fz = Z.clamp(0, 1) * (self.nZt - 1)
        fs = (s / self.s_max).clamp(0, 1) * (self.ns - 1)
        iz = fz.floor().clamp(max=self.nZt - 2).long()
        js = fs.floor().clamp(max=self.ns - 2).long()
        wz = fz - iz
        ws = fs - js
        f = tab.reshape(-1)
        b = iz * self.ns + js
        return ((1 - wz) * (1 - ws) * f[b] + wz * (1 - ws) * f[b + self.ns]
                + (1 - wz) * ws * f[b + 1] + wz * ws * f[b + self.ns + 1])

    def _cgrad(self, q, axis):
        """Derivada centrada de um campo de centro, ao longo de axis."""
        qp = _pad(q, axis)
        dist = _sl(self.Df[axis], axis, 0, -1) + _sl(self.Df[axis], axis, 1, None)
        return (_sl(qp, axis, 2, None) - _sl(qp, axis, 0, -2)) / dist

    def _svar(self, Z):
        g2 = self._cgrad(Z, 0) ** 2 + self._cgrad(Z, 1) ** 2 + self._cgrad(Z, 2) ** 2
        var = self.cfg.C_Z * self.delta ** 2 * g2
        return (var / (Z * (1 - Z) + 1e-8)).clamp(0, self.s_max)

    def _update_props(self, Z):
        s = self._svar(Z)
        self.s = s
        self.rho = self._lookup(self.tab_rho, Z, s)
        self.drho = self._lookup(self.tab_drho, Z, s)

    # --------------------------------------------------------- turbulência
    def _nu_t(self, u, v, w):
        """Viscosidade de submalha: Smagorinsky ou WALE (Nicoud & Ducros 1999)."""
        uc, vc, wc = _avg(u, 0), _avg(v, 1), _avg(w, 2)
        g = [[None] * 3 for _ in range(3)]
        for i, (q, qc) in enumerate(((u, uc), (v, vc), (w, wc))):
            for j in range(3):
                g[i][j] = _diff(q, j) / self.Dc[j] if j == i else self._cgrad(qc, j)
        SS = 0.0
        for i in range(3):
            for j in range(i, 3):
                Sij = 0.5 * (g[i][j] + g[j][i])
                SS = SS + (1.0 if i == j else 2.0) * Sij * Sij
        if self.cfg.sgs == "wale":
            g2 = [[sum(g[i][k] * g[k][j] for k in range(3)) for j in range(3)] for i in range(3)]
            tr = (g2[0][0] + g2[1][1] + g2[2][2]) / 3.0
            SdSd = 0.0
            for i in range(3):
                for j in range(i, 3):
                    Sd = 0.5 * (g2[i][j] + g2[j][i]) - (tr if i == j else 0.0)
                    SdSd = SdSd + (1.0 if i == j else 2.0) * Sd * Sd
            num = SdSd ** 1.5
            den = SS ** 2.5 + SdSd ** 1.25 + 1e-12
            return (self.cfg.Cw * self.delta) ** 2 * num / den
        return (self.cfg.Cs * self.delta) ** 2 * torch.sqrt(2.0 * SS)

    # ------------------------------------------------------------- momento
    def _spacings(self, comp, axis):
        """Espaçamento local h (advecção), distâncias h−/h+ e largura de controle (difusão)
        para a componente de velocidade comp ao longo de axis."""
        if comp == axis:   # variável nas faces ao longo de axis: vizinhas separadas por Δc
            dc = self.Dc[axis]
            hm = torch.cat([_sl(dc, axis, 0, 1), dc], dim=axis)
            hp = torch.cat([dc, _sl(dc, axis, -1, None)], dim=axis)
            return 0.5 * (hm + hp), hm, hp, self.Dcf[axis]
        dfa = self.Df[axis]
        return self.Dc[axis], _sl(dfa, axis, 0, -1), _sl(dfa, axis, 1, None), self.Dc[axis]

    def _adv_visc(self, comp, q, adv, mu_q, rho_q):
        """Upwind de 3ª ordem (= central de 4ª ordem + dissipação |a|h³/12 ∂⁴q) e ∇·(μ∇q)/ρ."""
        beta = self.cfg.upwind
        A = 0.0
        Vs = 0.0
        for axis, a in enumerate(adv):
            h, hm, hp, hc = self._spacings(comp, axis)
            qp = _pad(q, axis, 2)
            n = q.shape[axis]
            qm2, qm1 = _sl(qp, axis, 0, n), _sl(qp, axis, 1, n + 1)
            qp1, qp2 = _sl(qp, axis, 3, n + 3), _sl(qp, axis, 4, n + 4)
            dpos = (2 * qp1 + 3 * q - 6 * qm1 + qm2) / (6 * h)
            dneg = (-qp2 + 6 * qp1 - 3 * q - 2 * qm1) / (6 * h)
            d3 = torch.where(a >= 0, dpos, dneg)
            if beta > 0:
                d1 = torch.where(a >= 0, (q - qm1) / hm, (qp1 - q) / hp)
                d3 = (1 - beta) * d3 + beta * d1
            A = A + a * d3
            mp = _pad(mu_q, axis)
            mu_p = 0.5 * (mu_q + _sl(mp, axis, 2, None))
            mu_m = 0.5 * (mu_q + _sl(mp, axis, 0, -2))
            Vs = Vs + (mu_p * (qp1 - q) / hp - mu_m * (q - qm1) / hm) / hc
        return A - Vs / rho_q

    def _momentum_rhs(self, u, v, w, rho, Hc):
        cfg = self.cfg
        mu = rho * (self._nu_t(u, v, w) + 1.5e-5)
        uc, vc, wc = _avg(u, 0), _avg(v, 1), _avg(w, 2)
        ru, rv, rw = _to_faces(rho, 0), _to_faces(rho, 1), _to_faces(rho, 2)
        Fu = self._adv_visc(0, u, (u, _to_faces(vc, 0), _to_faces(wc, 0)), _to_faces(mu, 0), ru)
        Fv = self._adv_visc(1, v, (_to_faces(uc, 1), v, _to_faces(wc, 1)), _to_faces(mu, 1), rv)
        Fw = self._adv_visc(2, w, (_to_faces(uc, 2), _to_faces(vc, 2), w), _to_faces(mu, 2), rw)
        Fw = Fw + G * (rw - self.rho_inf) / rw                     # empuxo
        if cfg.baroclinic:                                          # −p̃∇(1/ρ) = H∇ρ/ρ
            Fu = Fu + _to_faces(Hc, 0) * _diff(_pad(rho, 0), 0) / self.Df[0] / ru
            Fv = Fv + _to_faces(Hc, 1) * _diff(_pad(rho, 1), 1) / self.Df[1] / rv
            Fw = Fw + _to_faces(Hc, 2) * _diff(_pad(rho, 2), 2) / self.Df[2] / rw
        # fonte de combustível: ρ Du/Dt += S_m (u_inj − u), com u_inj = (0, 0, u_jet)
        Fu = Fu + self.Sm_u * u / ru
        Fv = Fv + self.Sm_v * v / rv
        Fw = Fw - self.Sm_w * (cfg.u_jet - w) / rw
        if self.wind:                                               # tensão de parede (lei log)
            h0 = float(self.dc[2][0])
            u0, v0 = _sl(u, 2, 0, 1), _sl(v, 2, 0, 1)
            spd_u = torch.sqrt(u0 ** 2 + _to_faces(_sl(vc, 2, 0, 1), 0) ** 2) + 1e-6
            spd_v = torch.sqrt(_to_faces(_sl(uc, 2, 0, 1), 1) ** 2 + v0 ** 2) + 1e-6
            Fu = torch.cat([_sl(Fu, 2, 0, 1) + self.Cd_wall * spd_u * u0 / h0, _sl(Fu, 2, 1, None)], 2)
            Fv = torch.cat([_sl(Fv, 2, 0, 1) + self.Cd_wall * spd_v * v0 / h0, _sl(Fv, 2, 1, None)], 2)
        return Fu, Fv, Fw, mu

    # -------------------------------------------------- fração de mistura
    def _tvd_face(self, Z, a, axis):
        n = Z.shape[axis]
        Zp = _pad(Z, axis, 2)
        ZLL, ZL = _sl(Zp, axis, 0, n + 1), _sl(Zp, axis, 1, n + 2)
        ZR, ZRR = _sl(Zp, axis, 2, n + 3), _sl(Zp, axis, 3, n + 4)

        lim = self.cfg.limiter

        def vl(num, den):
            den = torch.where(den.abs() < 1e-12, torch.full_like(den, 1e-12), den)
            r = num / den
            if lim == "superbee":
                return torch.maximum(torch.clamp(2 * r, max=1.0), torch.clamp(r, max=2.0)).clamp(min=0.0)
            if lim == "mc":
                return torch.minimum(torch.minimum(2 * r, 0.5 * (1 + r)), torch.full_like(r, 2.0)).clamp(min=0.0)
            return (r + r.abs()) / (1 + r.abs())
        Zpos = ZL + 0.5 * vl(ZL - ZLL, ZR - ZL) * (ZR - ZL)
        Zneg = ZR + 0.5 * vl(ZR - ZRR, ZL - ZR) * (ZL - ZR)
        Zf = torch.where(a >= 0, Zpos, Zneg)
        lo, hi = _sl(Zf, axis, 0, 1), _sl(Zf, axis, -1, None)      # bordas: entra ar (Z = 0)
        lo = torch.where(_sl(a, axis, 0, 1) > 0, torch.zeros_like(lo), lo)
        hi = torch.where(_sl(a, axis, -1, None) < 0, torch.zeros_like(hi), hi)
        return torch.cat([lo, _sl(Zf, axis, 1, -1), hi], dim=axis)

    def _z_adv(self, Z, u, v, w):
        """−∇·(uZ) + Z∇·u com faces TVD."""
        adv = 0.0
        for axis, a in enumerate((u, v, w)):
            adv = adv + _diff(a * self._tvd_face(Z, a, axis), axis) / self.Dc[axis]
        return -adv + Z * self.divergence(u, v, w)

    def _z_diff_src(self, Z, mu):
        """∇·(ρD_t∇Z) + S_m(1 − Z)  [kg/(m3 s)]."""
        rhoD = mu / self.cfg.Sc_t
        dif = 0.0
        for axis in range(3):
            flux = _to_faces(rhoD, axis) * _diff(_pad(Z, axis), axis) / self.Df[axis]
            dif = dif + _diff(flux, axis) / self.Dc[axis]
        return dif + self.Sm * (1.0 - Z)

    def _divergence_target(self, rho, drho, DZ_rho):
        return self.Sm / rho - drho * DZ_rho / (rho * rho)

    # --------------------------------------------------------------- passo
    def _apply_bc(self, u, v, w):
        if self.wind:
            u = torch.cat([self.U_in * (1.0 + self.noise), u[1:]], 0)
        w = torch.cat([torch.zeros_like(w[:, :, :1]), w[:, :, 1:]], 2)
        return u, v, w

    def _compute_dt(self):
        uc, vc, wc = _avg(self.u, 0).abs(), _avg(self.v, 1).abs(), _avg(self.w, 2).abs()
        rate = (uc / self.Dc[0] + vc / self.Dc[1] + wc / self.Dc[2]).max().item()
        dt = self.cfg.cfl / max(rate, 1e-6)
        if self._numax > 0:
            dt = min(dt, 0.45 / self._numax)
        dt = min(dt, self.cfg.dt_max)
        return dt, rate * dt

    def step(self):
        t0 = time.perf_counter()
        dt, cfl = self._compute_dt()
        self.dt = dt
        cfg = self.cfg
        if self.wind and cfg.inflow_noise > 0:
            # ruído de entrada: AR(1) no tempo, suavizado na vertical
            eps = torch.randn_like(self.noise)
            eps = torch.nn.functional.avg_pool1d(eps.reshape(-1, 1, self.nz), 5, 1, 2).reshape(self.noise.shape)
            a = math.exp(-dt / 2.0)
            self.noise = a * self.noise + math.sqrt(1 - a * a) * cfg.inflow_noise * eps * 2.2
        if cfg.jet_noise > 0:
            # semente de turbulência do jato: impulsos horizontais aleatórios nas faces da fonte
            amp = cfg.jet_noise * cfg.u_jet ** 0.5 * 3.0 * math.sqrt(dt)
            self.u = self.u + self.src_u * torch.randn_like(self.u) * amp
            self.v = self.v + self.src_v * torch.randn_like(self.v) * amp
        u0, v0, w0, Z0, rho0, H0 = self.u, self.v, self.w, self.Z, self.rho, self.Hp
        # ---- estágio 1 (preditor)
        Fu, Fv, Fw, mu = self._momentum_rhs(u0, v0, w0, rho0, H0)
        RZ = self._z_adv(Z0, u0, v0, w0) + self._z_diff_src(Z0, mu) / rho0
        Z1 = (Z0 + dt * RZ).clamp(0, 1)
        self._update_props(Z1)
        rho1, drho1 = self.rho, self.drho
        D1 = self._divergence_target(rho1, drho1, self._z_diff_src(Z1, mu * (rho1 / rho0)))
        u1, v1, w1 = self._apply_bc(u0 - dt * Fu, v0 - dt * Fv, w0 - dt * Fw)
        u1, v1, w1, H1 = self.project(u1, v1, w1, D1, dt)
        # ---- estágio 2 (corretor)
        Fu, Fv, Fw, mu1 = self._momentum_rhs(u1, v1, w1, rho1, H1)
        RZ = self._z_adv(Z1, u1, v1, w1) + self._z_diff_src(Z1, mu1) / rho1
        Z2 = (0.5 * (Z0 + Z1 + dt * RZ)).clamp(0, 1)
        self._update_props(Z2)
        rho2, drho2 = self.rho, self.drho
        D2 = self._divergence_target(rho2, drho2, self._z_diff_src(Z2, mu1 * (rho2 / rho1)))
        u2, v2, w2 = self._apply_bc(0.5 * (u0 + u1 - dt * Fu), 0.5 * (v0 + v1 - dt * Fv),
                                    0.5 * (w0 + w1 - dt * Fw))
        u2, v2, w2, H2 = self.project(u2, v2, w2, D2, 0.5 * dt)
        self.u, self.v, self.w, self.Z, self.Hp = u2, v2, w2, Z2, H2
        self._numax = ((mu1 / rho1) / cfg.Sc_t * 2.0 * self.inv_d2).max().item()
        if not math.isfinite(self._numax) or not torch.isfinite(self.w).all():
            raise FloatingPointError(f"a simulação divergiu em t = {self.time:.2f} s")
        self.time += dt
        self.step_n += 1
        if self.dev.type == "cuda":
            torch.cuda.synchronize()
        self.wall += time.perf_counter() - t0
        self._last_cfl = cfl
        return dt

    # ---------------------------------------------------------- diagnósticos
    def fields(self):
        """T̃, emissão luminosa (não normalizada) e máscara da chama (Z̃ ≥ Z_st)."""
        T = self._lookup(self.tab_T, self.Z, self.s)
        e = self._lookup(self.tab_e, self.Z, self.s)
        return T, e, self.Z >= self.Z_st

    def ground_flux(self, e):
        """Fluxo incidente no solo (pior orientação, seção 3.8): soma volumétrica dos emissores
        com transmissividade de Wayne, potência total escalada para X_rad·Q (como no FDS)."""
        cfg = self.cfg
        P = (e * self.vol).reshape(-1)
        keep = P > 1e-4 * P.max().clamp(min=1e-30)
        P, xyz = P[keep], self.cell_xyz[keep]
        if len(P) == 0:
            return torch.zeros(len(self.receivers), dtype=self.ft, device=self.dev)
        # agrupa emissores em blocos (centroide ponderado pela potência)
        key = torch.floor(xyz / cfg.emitter_bin).long()
        key = key - key.min(0).values
        dims = key.max(0).values + 1
        lin = (key[:, 0] * dims[1] + key[:, 1]) * dims[2] + key[:, 2]
        uniq, inv = torch.unique(lin, return_inverse=True)
        Pb = torch.zeros(len(uniq), dtype=self.ft, device=self.dev).index_add_(0, inv, P)
        Xb = torch.zeros(len(uniq), 3, dtype=self.ft, device=self.dev).index_add_(0, inv, P[:, None] * xyz)
        Xb = Xb / Pb[:, None]
        scale = cfg.X_rad * self.Q / Pb.sum()
        psat = math.exp(20.386 - 5132.0 / cfg.T_inf)
        V = torch.zeros_like(self.receivers)
        for i0 in range(0, len(Pb), 2048):
            dv = Xb[None, i0:i0 + 2048] - self.receivers[:, None]
            r = torch.linalg.norm(dv, dim=-1).clamp(min=0.5)
            lw = torch.log10(cfg.RH * r * psat * 2.88651e2 / cfg.T_inf)
            lc = torch.log10(r * 273.0 / cfg.T_inf)
            tau = (1.006 - 0.01171 * lw - 0.02368 * lw ** 2 - 0.03188 * lc + 0.001164 * lc ** 2).clamp(0, 1)
            wgt = Pb[None, i0:i0 + 2048] * tau / (4 * math.pi * r ** 3)
            V += (wgt[..., None] * dv).sum(1)
        return torch.linalg.norm(V, dim=1) * scale

    def _centroid_tilt(self, mask):
        if not mask.any():
            return float("nan")
        c = self.cell_xyz[mask.reshape(-1)].mean(0).cpu().numpy() - self.tip
        return math.degrees(math.atan2(math.hypot(c[0], c[1]), max(c[2], 1e-6)))

    def diagnostics(self, averaging: bool):
        T, e, mask = self.fields()
        q = self.ground_flux(e)
        self.q_inst = q
        L = torch.where(mask, self.dist_tip, torch.zeros_like(self.dist_tip)).max().item()
        h = self.history
        h["t"].append(self.time); h["L"].append(L); h["tilt"].append(self._centroid_tilt(mask))
        h["qmax"].append(q.max().item()); h["Tmax"].append(T.max().item()); h["dt"].append(self.dt)
        h["cfl"].append(self._last_cfl)
        if averaging:
            self.avg_n += 1.0
            self.avg_T += T
            self.avg_I += mask.to(self.ft)
            self.avg_q += q
        return T, e, mask, q

    def mean_flame_length(self, threshold: float = 0.5):
        if self.avg_n == 0:
            return float("nan")
        I = self.avg_I / self.avg_n
        return torch.where(I >= threshold, self.dist_tip, torch.zeros_like(self.dist_tip)).max().item()

    def mean_tilt(self, threshold: float = 0.5):
        if self.avg_n == 0:
            return float("nan")
        return self._centroid_tilt(self.avg_I / self.avg_n >= threshold)

    def q_grid(self, q):
        return q.reshape(len(self.rec_x), len(self.rec_y)).cpu().numpy()

    def slice_y0(self, f):
        return f[:, self.j_mid, :].float().cpu().numpy()

    def mass_balance(self):
        """Massa no domínio e vazão volumétrica líquida pelas fronteiras."""
        m = (self.rho * self.vol).sum().item()
        A_x = torch.outer(self._t(self.dc[1]), self._t(self.dc[2]))
        A_y = torch.outer(self._t(self.dc[0]), self._t(self.dc[2]))
        A_z = torch.outer(self._t(self.dc[0]), self._t(self.dc[1]))
        out = ((self.u[-1] - self.u[0]) * A_x).sum() + ((self.v[:, -1] - self.v[:, 0]) * A_y).sum() \
            + (self.w[:, :, -1] * A_z).sum()
        return m, out.item()

    def summary(self) -> str:
        dmin = min(float(d.min()) for d in self.dc)
        dmax = max(float(d.max()) for d in self.dc)
        return (f"{self.nx}×{self.ny}×{self.nz} = {self.n_cells / 1e6:.2f} M células · Δ {dmin:.2f}–{dmax:.2f} m · "
                f"fonte {self.src_area:.2f} m² · {self.dev.type.upper()}")

In [ ]:
%%writefile flarekit/dashboard.py
"""Painel ao vivo da LES e gravação do vídeo (MP4) quadro a quadro.

O mesmo quadro desenhado para o vídeo é mostrado no notebook enquanto o solver roda,
de modo que os gráficos acompanham a solução em tempo real.
"""
from __future__ import annotations

import io
import math
import time

import numpy as np
from matplotlib import colors as mcolors
from matplotlib.animation import FFMpegWriter
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

from . import semiempirical as se

BG, PANEL, FG, MUTED, GRID = "#0d1117", "#131a22", "#e6edf3", "#8b98a5", "#263241"
ACCENT, LES_C, CHAM_C, PT_C = "#f0a35e", "#ffb347", "#7fd1ff", "#b9c2cc"
API_LEVELS = [(1.58, "#6cc08f"), (4.73, "#e0b24a"), (6.31, "#f08a4b"), (9.46, "#f2766b")]


def _style(ax, title=None):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.xaxis.label.set_color(MUTED)
    ax.yaxis.label.set_color(MUTED)
    if title:
        ax.set_title(title, color=FG, fontsize=11, loc="left", pad=6, fontweight="bold")


def frustum_outline_xz(ch: se.Chamberlain, tip):
    """Trapézio do frustum de Chamberlain no plano y = 0 (vento em +x)."""
    base, ax, e1, _ = se.frustum_axis(ch, np.asarray(tip, float))
    top = base + ch.R_l * ax
    pts = [base - ch.W1 / 2 * e1, top - ch.W2 / 2 * e1, top + ch.W2 / 2 * e1, base + ch.W1 / 2 * e1,
           base - ch.W1 / 2 * e1]
    return np.array([[p[0], p[2]] for p in pts])


class LiveDashboard:
    def __init__(self, les, sc: se.Scenario, title: str, view_x=(-25.0, 95.0), view_z=(0.0, 80.0),
                 dpi: int = 120, live: bool = True):
        self.les, self.sc, self.live = les, sc, live
        self.fig = Figure(figsize=(16, 9), dpi=dpi, facecolor=BG)
        FigureCanvasAgg(self.fig)
        gs = GridSpec(3, 5, figure=self.fig, left=0.045, right=0.985, top=0.865, bottom=0.07,
                      hspace=0.45, wspace=0.42)
        self.ax_s = self.fig.add_subplot(gs[0:2, 0:3])
        self.ax_m = self.fig.add_subplot(gs[0:2, 3:5])
        self.ax_l = self.fig.add_subplot(gs[2, 0:2])
        self.ax_t = self.fig.add_subplot(gs[2, 2:4])
        self.ax_i = self.fig.add_subplot(gs[2, 4])
        self.fig.text(0.045, 0.952, title, color=FG, fontsize=17, fontweight="bold")
        self.sub = self.fig.text(0.045, 0.918, "", color=MUTED, fontsize=10.5)
        self.clock = self.fig.text(0.985, 0.95, "", color=ACCENT, fontsize=20, ha="right", fontweight="bold",
                                   family="monospace")
        ch = sc.cham if les.wind else sc.cham0
        self.ch = ch

        # ---- corte vertical y = 0: temperatura + chama + frustum de Chamberlain
        ax = self.ax_s
        _style(ax, "Temperatura no plano y = 0  ·  contorno Z̃ = Z_st (chama)")
        T0 = les.slice_y0(les.fields()[0])
        self.mesh = ax.pcolormesh(les.xf, les.zf, T0.T, cmap="inferno", vmin=290, vmax=1800, shading="flat",
                                  rasterized=True)
        cb = self.fig.colorbar(self.mesh, ax=ax, pad=0.01, fraction=0.035)
        cb.set_label("T [K]", color=MUTED); cb.ax.tick_params(colors=MUTED, labelsize=8)
        cb.outline.set_edgecolor(GRID)
        ax.add_patch(Rectangle((-0.6, 0), 1.2, les.tip[2], color="#5c6773", zorder=3))
        fr = frustum_outline_xz(ch, les.tip)
        ax.plot(fr[:, 0], fr[:, 1], ls="--", color=CHAM_C, lw=1.6, zorder=4, label="Chamberlain (1987)")
        self.flame_c = None
        ax.set_xlim(*view_x); ax.set_ylim(*view_z); ax.set_aspect("equal")
        ax.set_xlabel("x [m] (vento →)"); ax.set_ylabel("z [m]")
        ax.legend(loc="upper right", fontsize=9, facecolor=PANEL, edgecolor=GRID, labelcolor=FG)
        if les.wind:
            ax.annotate("", xy=(view_x[0] + 12, view_z[1] - 6), xytext=(view_x[0] + 2, view_z[1] - 6),
                        arrowprops=dict(arrowstyle="-|>", color=CHAM_C, lw=2))
            ax.text(view_x[0] + 2, view_z[1] - 11, f"{les.cfg.u_ref:.1f} m/s", color=CHAM_C, fontsize=9)

        # ---- mapa de radiação no solo
        ax = self.ax_m
        _style(ax, "Radiação incidente no solo (1,5 m) [kW/m²]")
        q0 = les.q_grid(les.q_inst) / 1e3
        self.norm = mcolors.PowerNorm(0.6, vmin=0.0, vmax=10.0)
        self.qmesh = ax.pcolormesh(les.rec_x, les.rec_y, q0.T, cmap="magma", norm=self.norm, shading="nearest",
                                   rasterized=True)
        cb = self.fig.colorbar(self.qmesh, ax=ax, pad=0.01, fraction=0.04)
        cb.ax.tick_params(colors=MUTED, labelsize=8); cb.outline.set_edgecolor(GRID)
        for lev, c in API_LEVELS:
            cb.ax.axhline(lev, color=c, lw=2)
        self.q_cont = None
        ax.plot([0], [0], marker="^", color="white", ms=9, mec="black", zorder=5)
        ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
        ax.set_xlim(les.rec_x[0], les.rec_x[-1]); ax.set_ylim(les.rec_y[0], les.rec_y[-1])
        self.map_note = ax.text(0.02, 0.03, "", transform=ax.transAxes, color=FG, fontsize=9)
        handles = [Line2D([], [], color=c, lw=2, label=f"{lev:.2f}") for lev, c in API_LEVELS]
        ax.legend(handles=handles, title="API 521", loc="upper right", fontsize=8, title_fontsize=8,
                  facecolor=PANEL, edgecolor=GRID, labelcolor=FG).get_title().set_color(FG)

        # ---- fluxo ao longo do vento (y = 0) vs modelos semi-empíricos
        ax = self.ax_l
        _style(ax, "Fluxo no solo ao longo do vento (y = 0)")
        jl = int(np.argmin(np.abs(les.rec_y)))
        self.jl = jl
        xr = les.rec_x
        rec = np.stack([xr, np.zeros_like(xr), np.full_like(xr, les.cfg.receiver_z)], 1)
        self.q_cham_line = se.q_chamberlain(rec, ch, les.tip, sc.T_inf, sc.RH) / 1e3
        center = se.flame_center_chamberlain(ch, les.tip)
        self.q_pt_line = se.q_point_source(rec, center, les.cfg.X_rad, sc.Q, sc.T_inf, sc.RH) / 1e3
        ax.plot(xr, self.q_cham_line, "--", color=CHAM_C, lw=1.8, label="Chamberlain (frustum)")
        ax.plot(xr, self.q_pt_line, ":", color=PT_C, lw=1.8, label="Fonte pontual (API 521)")
        self.l_inst, = ax.plot(xr, np.zeros_like(xr), color=LES_C, lw=1.0, alpha=0.45, label="LES instantâneo")
        self.l_mean, = ax.plot(xr, np.zeros_like(xr), color=LES_C, lw=2.6, label="LES média")
        for lev, c in API_LEVELS:
            ax.axhline(lev, color=c, lw=0.8, alpha=0.6)
        ymax = 1.6 * max(self.q_cham_line.max(), self.q_pt_line.max())
        ax.set_ylim(0, ymax); ax.set_xlim(xr[0], xr[-1])
        ax.set_xlabel("x [m]"); ax.set_ylabel("q'' [kW/m²]")
        ax.grid(color=GRID, lw=0.5)
        ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, ncol=2, loc="upper right")

        # ---- comprimento da chama no tempo
        ax = self.ax_t
        _style(ax, "Comprimento da chama no tempo")
        refL = ch.L_b
        ax.axhline(refL, color=CHAM_C, ls="--", lw=1.6, label=f"Chamberlain {refL:.1f} m")
        if not les.wind:
            ax.axhline(sc.L_api, color=PT_C, ls=":", lw=1.6, label=f"API 521 {sc.L_api:.1f} m")
        self.t_inst, = ax.plot([], [], color=LES_C, lw=1.0, alpha=0.5, label="LES instantâneo")
        self.t_mean, = ax.plot([], [], color=LES_C, lw=2.6, label="LES média (I = 0,5)")
        ax.set_ylim(0, 1.8 * refL); ax.set_xlabel("t [s]"); ax.set_ylabel("L [m]")
        ax.grid(color=GRID, lw=0.5)
        ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, loc="upper right")
        self.mean_hist_t, self.mean_hist_L = [], []

        # ---- painel de informações
        ax = self.ax_i
        ax.set_facecolor(PANEL); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_color(GRID)
        self.info = ax.text(0.04, 0.96, "", transform=ax.transAxes, va="top", color=FG, fontsize=8.4,
                            family="monospace", linespacing=1.45)
        self.writer = None
        self.handle = None

    # ------------------------------------------------------------------ vídeo
    def start_video(self, path: str, fps: int = 24):
        self.writer = FFMpegWriter(fps=fps, bitrate=9000, codec="libx264",
                                   extra_args=["-pix_fmt", "yuv420p", "-preset", "medium"])
        self.writer.setup(self.fig, path, dpi=self.fig.dpi)
        self.fps = fps

    def finish_video(self, hold_s: float = 2.5):
        if self.writer is not None:
            for _ in range(int(hold_s * self.fps)):
                self.writer.grab_frame(facecolor=BG)
            self.writer.finish()
            self.writer = None

    # --------------------------------------------------------------- quadro
    def update(self, T, mask, q, averaging: bool, sub: str = ""):
        les, ch = self.les, self.ch
        self.mesh.set_array(les.slice_y0(T).T.ravel())
        if self.flame_c is not None:
            self.flame_c.remove()
        Zs = les.slice_y0(les.Z)
        self.flame_c = self.ax_s.contour(les.xc, les.zc, Zs.T, levels=[les.Z_st], colors=["#4dd0e1"],
                                         linewidths=1.6)
        qg = les.q_grid(q) / 1e3
        qm = les.q_grid(les.avg_q / les.avg_n) / 1e3 if les.avg_n > 0 else None
        show = qm if (averaging and qm is not None) else qg
        self.qmesh.set_array(show.T.ravel())
        if self.q_cont is not None:
            self.q_cont.remove()
        levels = [lev for lev, _ in API_LEVELS if lev < show.max()]
        cols = [c for lev, c in API_LEVELS if lev < show.max()]
        self.q_cont = self.ax_m.contour(les.rec_x, les.rec_y, show.T, levels=levels, colors=cols, linewidths=1.8) \
            if levels else None
        self.map_note.set_text("média temporal" if (averaging and qm is not None) else "instantâneo")
        self.l_inst.set_ydata(qg[:, self.jl])
        if qm is not None:
            self.l_mean.set_ydata(qm[:, self.jl])
        h = les.history
        self.t_inst.set_data(h["t"], h["L"])
        if averaging and les.avg_n > 0:
            self.mean_hist_t.append(les.time)
            self.mean_hist_L.append(les.mean_flame_length())
            self.t_mean.set_data(self.mean_hist_t, self.mean_hist_L)
        self.ax_t.set_xlim(0, max(5.0, les.time * 1.05))
        Lm = les.mean_flame_length()
        tm = les.mean_tilt()
        qmax_m = float(qm.max()) if qm is not None else float("nan")
        fmt = lambda v, f="{:.1f}": "  —  " if not math.isfinite(v) else f.format(v)  # noqa: E731
        lines = [
            f"t     {les.time:6.2f} s",
            f"passo {les.step_n:6d}",
            f"Δt    {les.dt * 1e3:6.1f} ms  CFL {les._last_cfl:.2f}",
            f"wall  {les.wall:6.0f} s",
            "",
            "VALIDAÇÃO     LES   Cham.",
            f"L [m]      {fmt(Lm):>6} {ch.L_b:6.1f}",
            f"incl. [°]  {fmt(tm):>6} {ch.alpha:6.1f}",
            f"q máx      {fmt(qmax_m, '{:.2f}'):>6} {self.q_cham_line.max():6.2f}",
            "(q em kW/m², médias)",
            "",
            f"X_rad = {les.cfg.X_rad:.3f} (F_s)",
        ]
        self.info.set_text("\n".join(lines))
        self.clock.set_text(f"t = {les.time:5.1f} s")
        self.sub.set_text(sub)
        if self.writer is not None:
            self.writer.grab_frame(facecolor=BG)
        else:
            self.fig.canvas.draw()
        if self.live:
            self._show()

    def _show(self):
        try:
            from IPython.display import Image, display
        except ImportError:
            return
        buf = io.BytesIO()
        from PIL import Image as PILImage
        rgba = np.asarray(self.fig.canvas.buffer_rgba())
        PILImage.fromarray(rgba[..., :3]).save(buf, format="JPEG", quality=82)
        img = Image(data=buf.getvalue(), format="jpeg")
        if self.handle is None:
            self.handle = display(img, display_id=True)
        else:
            self.handle.update(img)

    def save_png(self, path: str):
        self.fig.savefig(path, dpi=self.fig.dpi, facecolor=BG)


def run_live(les, sc, title: str, t_end: float, t_avg: float, frame_dt: float = 0.1,
             video_path: str | None = "flare_les.mp4", live: bool = True, fps: int = 24,
             max_wall_s: float | None = None, log_every: int = 0):
    """Roda a LES até t_end, atualizando o painel a cada frame_dt (tempo físico) e gravando o vídeo."""
    dash = LiveDashboard(les, sc, title, live=live)
    if video_path:
        dash.start_video(video_path, fps=fps)
    sub = (f"{les.summary()}  ·  Smagorinsky + Z̃/β-PDF (equilíbrio) + fuligem  ·  "
           f"média a partir de t = {t_avg:.0f} s")
    next_frame = 0.0
    t_start = time.time()
    try:
        while les.time < t_end:
            les.step()
            if les.time >= next_frame:
                averaging = les.time >= t_avg
                T, e, mask, q = les.diagnostics(averaging)
                dash.update(T, mask, q, averaging, sub)
                next_frame += frame_dt
                if log_every and len(les.history["t"]) % log_every == 0:
                    print(f"t = {les.time:6.2f} s · L = {les.history['L'][-1]:5.1f} m · "
                          f"{les.wall / les.step_n * 1e3:5.0f} ms/passo", flush=True)
            if max_wall_s and time.time() - t_start > max_wall_s:
                print(f"Tempo de parede máximo atingido em t = {les.time:.1f} s.")
                break
    finally:
        dash.finish_video()
    return dash

In [ ]:
%%writefile flarekit/safety.py
"""Critérios de segurança (seção 5 do documento): zonas do API 521, dose, probits, fuga e
temperatura de superfícies expostas."""
from __future__ import annotations

import math

import numpy as np
from scipy.optimize import brentq
from scipy.special import erf

from .props import SIGMA

API_521 = [
    (9.46, "máximo com acesso de pessoal (poucos segundos, só fuga)"),
    (6.31, "ações de emergência de até ~30 s, sem abrigo"),
    (4.73, "ações de emergência de 2–3 min, sem abrigo"),
    (1.58, "exposição contínua com roupa adequada"),
]

PROBITS = {
    "queimadura 1º grau (Tsao & Perry)": (-39.83, 3.0186),
    "queimadura 2º grau (Tsao & Perry)": (-43.14, 3.0186),
    "fatalidade (TNO Green Book)": (-36.38, 2.56),
    "fatalidade (Eisenberg)": (-38.48, 2.56),
}


def probability(Y):
    return 0.5 * (1.0 + erf((np.asarray(Y) - 5.0) / math.sqrt(2.0)))


def dose_escape(q0_W: float, r0: float, t_react: float, u_esc: float, q_safe_W: float = 1580.0) -> dict:
    """Dose com reação e fuga radial (dedução da seção 5.2):
    D = q0^(4/3) {t_r + (3/5)(r0/u)[1 − (q_s/q0)^(5/6)]}."""
    frac = 0.0 if q0_W <= q_safe_W else 1.0 - (q_safe_W / q0_W) ** (5.0 / 6.0)
    t_eff = t_react + 0.6 * r0 / u_esc * frac
    D = t_eff * q0_W ** (4.0 / 3.0)
    out = {"t_eff_s": t_eff, "dose_TDU": D / 1e4}
    for name, (a, b) in PROBITS.items():
        out[name] = float(probability(a + b * math.log(D)))
    return out


def steel_temperature(q_inc_W: float, T_inf: float = 298.15, alpha: float = 0.9, eps: float = 0.9,
                      h: float = 15.0) -> float:
    """Placa fina isolada atrás, regime permanente: α q = h(T−T∞) + εσ(T⁴−T∞⁴). Retorna °C."""
    f = lambda T: alpha * q_inc_W - h * (T - T_inf) - eps * SIGMA * (T ** 4 - T_inf ** 4)  # noqa: E731
    return brentq(f, T_inf, 3000.0) - 273.15


def zone_extents(q_kW: np.ndarray, rec_x: np.ndarray, rec_y: np.ndarray, stack=(0.0, 0.0)) -> list[dict]:
    """Para cada nível do API 521: área no solo e distância máxima a partir do pé do flare."""
    dx = float(rec_x[1] - rec_x[0]) if len(rec_x) > 1 else 1.0
    dy = float(rec_y[1] - rec_y[0]) if len(rec_y) > 1 else 1.0
    RX, RY = np.meshgrid(rec_x, rec_y, indexing="ij")
    R = np.hypot(RX - stack[0], RY - stack[1])
    out = []
    for lev, desc in API_521:
        m = q_kW >= lev
        out.append({"nivel_kW_m2": lev, "descricao": desc, "area_m2": float(m.sum() * dx * dy),
                    "raio_max_m": float(R[m].max()) if m.any() else 0.0,
                    "x_min_m": float(RX[m].min()) if m.any() else float("nan"),
                    "x_max_m": float(RX[m].max()) if m.any() else float("nan")})
    return out

In [ ]:
%%writefile flarekit/report.py
"""Resultados finais: tabela de validação, figura-resumo (16:9) e composição do vídeo final."""
from __future__ import annotations

import math
import shutil
import subprocess

import numpy as np
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec

from . import safety
from . import semiempirical as se
from .dashboard import API_LEVELS, BG, CHAM_C, FG, GRID, LES_C, MUTED, PANEL, PT_C, _style, frustum_outline_xz


def validation_rows(les, sc: se.Scenario, refs: dict) -> list[dict]:
    ch = sc.cham if les.wind else sc.cham0
    Lm, tm = les.mean_flame_length(), les.mean_tilt()
    qm = (les.avg_q / max(les.avg_n, 1)).max().item() / 1e3
    xr = les.rec_x
    rec = np.stack([xr, np.zeros_like(xr), np.full_like(xr, les.cfg.receiver_z)], 1)
    q_ch = se.q_chamberlain(rec, ch, les.tip, sc.T_inf, sc.RH).max() / 1e3
    rows = [
        {"grandeza": "Comprimento da chama L [m]", "LES": Lm, "referência": ch.L_b, "fonte": "Chamberlain (1987)"},
        {"grandeza": "Inclinação da chama [°]", "LES": tm, "referência": ch.alpha, "fonte": "Chamberlain (1987)"},
        {"grandeza": "Fluxo máximo no solo [kW/m²]", "LES": qm, "referência": q_ch, "fonte": "Chamberlain (frustum)"},
    ]
    if not les.wind:
        rows += [
            {"grandeza": "L vs API 521 [m]", "LES": Lm, "referência": sc.L_api, "fonte": "API 521 (Beychok)"},
            {"grandeza": "L vs Delichatsios [m]", "LES": Lm, "referência": refs["delichatsios"]["L"],
             "fonte": "Delichatsios (1993)"},
            {"grandeza": "L vs Heskestad [m]", "LES": Lm, "referência": refs["L_heskestad"], "fonte": "Heskestad"},
        ]
    for r in rows:
        r["erro_%"] = 100.0 * (r["LES"] - r["referência"]) / r["referência"] if r["referência"] else float("nan")
    return rows


def print_validation(rows):
    print(f"{'Grandeza':34s} {'LES':>8s} {'Ref.':>8s} {'Erro':>8s}  Fonte")
    for r in rows:
        print(f"{r['grandeza']:34s} {r['LES']:8.2f} {r['referência']:8.2f} {r['erro_%']:7.1f}%  {r['fonte']}")


def summary_figure(les, sc: se.Scenario, refs: dict, path: str, title: str, dpi: int = 120):
    ch = sc.cham if les.wind else sc.cham0
    fig = Figure(figsize=(16, 9), dpi=dpi, facecolor=BG)
    FigureCanvasAgg(fig)
    gs = GridSpec(2, 3, figure=fig, left=0.045, right=0.985, top=0.86, bottom=0.07, hspace=0.32, wspace=0.32,
                  width_ratios=[1.0, 1.15, 1.0])
    fig.text(0.045, 0.952, title, color=FG, fontsize=17, fontweight="bold")
    fig.text(0.045, 0.918, f"Médias de {les.avg_n:.0f} quadros (t ≥ início da média) · {les.summary()}",
             color=MUTED, fontsize=10.5)

    # temperatura média + chama média (intermitência 0,5) + Chamberlain
    ax = fig.add_subplot(gs[:, 0])
    _style(ax, "Temperatura média e chama média (I = 0,5)")
    Tm = les.slice_y0(les.avg_T / les.avg_n)
    Im = les.slice_y0(les.avg_I / les.avg_n)
    m = ax.pcolormesh(les.xf, les.zf, Tm.T, cmap="inferno", vmin=290, vmax=1500, shading="flat", rasterized=True)
    ax.contour(les.xc, les.zc, Im.T, levels=[0.5], colors=["#4dd0e1"], linewidths=2)
    fr = frustum_outline_xz(ch, les.tip)
    ax.plot(fr[:, 0], fr[:, 1], "--", color=CHAM_C, lw=1.8, label="Chamberlain (1987)")
    ax.plot([], [], color="#4dd0e1", lw=2, label="LES (I = 0,5)")
    from matplotlib.patches import Rectangle
    ax.add_patch(Rectangle((-0.6, 0), 1.2, les.tip[2], color="#5c6773"))
    ax.set_xlim(-20, 70); ax.set_ylim(0, 85); ax.set_aspect("equal")
    ax.set_xlabel("x [m]"); ax.set_ylabel("z [m]")
    ax.legend(fontsize=9, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, loc="upper right")
    cb = fig.colorbar(m, ax=ax, fraction=0.04, pad=0.01)
    cb.ax.tick_params(colors=MUTED, labelsize=8); cb.outline.set_edgecolor(GRID)

    # radiação média no solo e zonas do API 521
    ax = fig.add_subplot(gs[0, 1:])
    _style(ax, "Radiação média no solo e zonas do API 521")
    qm = les.q_grid(les.avg_q / les.avg_n) / 1e3
    from matplotlib import colors as mcolors
    mm = ax.pcolormesh(les.rec_x, les.rec_y, qm.T, cmap="magma", norm=mcolors.PowerNorm(0.6, 0, 10),
                       shading="nearest", rasterized=True)
    levels = [lev for lev, _ in API_LEVELS if lev < qm.max()]
    cols = [c for lev, c in API_LEVELS if lev < qm.max()]
    if levels:
        ax.contour(les.rec_x, les.rec_y, qm.T, levels=levels, colors=cols, linewidths=2)
    ax.plot([0], [0], marker="^", color="white", ms=9, mec="black")
    ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    cb = fig.colorbar(mm, ax=ax, fraction=0.03, pad=0.01)
    cb.ax.tick_params(colors=MUTED, labelsize=8); cb.outline.set_edgecolor(GRID)
    zones = safety.zone_extents(qm, les.rec_x, les.rec_y)

    # validação
    ax = fig.add_subplot(gs[1, 1])
    _style(ax, "Validação (LES vs. modelos de referência)")
    rows = validation_rows(les, sc, refs)
    short = {"Comprimento da chama L [m]": "L chama [m]", "Inclinação da chama [°]": "Inclinação [°]",
             "Fluxo máximo no solo [kW/m²]": "q máx [kW/m²]", "L vs API 521 [m]": "L × API [m]",
             "L vs Delichatsios [m]": "L × Delichatsios", "L vs Heskestad [m]": "L × Heskestad"}
    names = [short.get(r["grandeza"], r["grandeza"]) for r in rows]
    y = np.arange(len(rows))
    ax.barh(y + 0.2, [r["LES"] for r in rows], 0.38, color=LES_C, label="LES")
    ax.barh(y - 0.2, [r["referência"] for r in rows], 0.38, color=CHAM_C, label="Referência")
    for i, r in enumerate(rows):
        ax.text(max(r["LES"], r["referência"]) * 1.02, i, f"{r['erro_%']:+.0f}%", va="center", color=FG, fontsize=9)
    ax.set_yticks(y); ax.set_yticklabels(names, color=FG, fontsize=9)
    ax.invert_yaxis()
    ax.legend(fontsize=8, facecolor=PANEL, edgecolor=GRID, labelcolor=FG, loc="lower right")
    ax.grid(color=GRID, lw=0.5, axis="x")

    # segurança no ponto mais exposto
    ax = fig.add_subplot(gs[1, 2])
    ax.set_facecolor(PANEL); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_color(GRID)
    i, j = np.unravel_index(np.argmax(qm), qm.shape)
    xq, yq, q0 = les.rec_x[i], les.rec_y[j], qm[i, j]
    center = se.flame_center_chamberlain(ch, les.tip)
    r0 = float(np.linalg.norm(center - np.array([xq, yq, les.cfg.receiver_z])))
    d1 = safety.dose_escape(q0 * 1e3, r0, 5.0, 2.5)
    d2 = safety.dose_escape(q0 * 1e3, r0, 30.0, 2.5)
    lines = [
        "PONTO MAIS EXPOSTO NO SOLO",
        f"x = {xq:.0f} m, y = {yq:.0f} m: q = {q0:.2f} kW/m²",
        f"T aço exposto (regime): {safety.steel_temperature(q0 * 1e3):.0f} °C",
        "",
        "Dose com fuga a 2,5 m/s:",
        f" reage em 5 s : {d1['dose_TDU']:5.0f} TDU",
        f"   1º grau {100 * d1['queimadura 1º grau (Tsao & Perry)']:5.1f}%"
        f"  2º grau {100 * d1['queimadura 2º grau (Tsao & Perry)']:5.2f}%",
        f" reage em 30 s: {d2['dose_TDU']:5.0f} TDU",
        f"   1º grau {100 * d2['queimadura 1º grau (Tsao & Perry)']:5.1f}%"
        f"  2º grau {100 * d2['queimadura 2º grau (Tsao & Perry)']:5.2f}%",
        f"   fatal (TNO) {100 * d2['fatalidade (TNO Green Book)']:5.2f}%",
        "",
        "ZONAS API 521     área [m²]  alcance [m]",
    ] + [f"{z['nivel_kW_m2']:5.2f} kW/m²  {z['area_m2']:9.0f}  {z['raio_max_m']:9.0f}" for z in zones]
    ax.text(0.04, 0.96, "\n".join(lines), transform=ax.transAxes, va="top", color=FG, fontsize=9,
            family="monospace", linespacing=1.5)
    fig.savefig(path, dpi=dpi, facecolor=BG)
    return fig, rows, zones


def compose_video(main_mp4: str, summary_png: str, out_mp4: str, seconds: float = 6.0, fps: int = 24) -> str:
    """Anexa a figura-resumo ao fim do vídeo da simulação (ffmpeg)."""
    ff = shutil.which("ffmpeg")
    if ff is None:
        raise RuntimeError("ffmpeg não encontrado")
    cmd = [ff, "-y", "-i", main_mp4, "-loop", "1", "-t", f"{seconds}", "-framerate", f"{fps}", "-i", summary_png,
           "-filter_complex",
           "[0:v]scale=1920:1080,setsar=1,fps=%d[a];[1:v]scale=1920:1080,setsar=1,fps=%d,format=yuv420p[b];"
           "[a][b]concat=n=2:v=1:a=0[v]" % (fps, fps),
           "-map", "[v]", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "20", out_mp4]
    subprocess.run(cmd, check=True, capture_output=True)
    return out_mp4

In [ ]:
import importlib, flarekit
print('flarekit pronto')

## 1 · Cenário e referências

In [ ]:
#@title 1 · Cenário e modelos de referência (API 521, Chamberlain, correlações)
import numpy as np
from flarekit.props import PROPANE, stoichiometry, state_relation, beta_pdf_table
from flarekit.semiempirical import Scenario

# --- cenário (o mesmo do exemplo numérico do documento) -----------------------------
VENTO = 8.9    #@param {type:"number"}
ALTURA = 30.0  #@param {type:"number"}
VAZAO = 12.6   #@param {type:"number"}
sc = Scenario(PROPANE, mdot=VAZAO, T_j=311.0, mach=0.5, H=ALTURA, u_w=VENTO, T_inf=298.15, RH=0.5)

tip, ch, ch0 = sc.tip, sc.cham, sc.cham0
print(f"Tip: u_j = {tip['u_j']:.1f} m/s · d = {tip['d_j']:.3f} m · Q = {sc.Q/1e6:.0f} MW")
print(f"API 521 (Beychok):      L = {sc.L_api:5.1f} m")
print(f"Chamberlain, sem vento: L = {ch0.L_b:5.1f} m")
print(f"Chamberlain, {sc.u_w} m/s: L = {ch.L_b:5.1f} m · α = {ch.alpha:4.1f}° · b = {ch.b:.1f} m · "
      f"W1 = {ch.W1:.2f} m · W2 = {ch.W2:.1f} m · F_s = {ch.F_s:.3f} · SEP = {ch.SEP/1e3:.0f} kW/m²")

## 2 · Termoquímica

In [ ]:
#@title 2 · Termoquímica: equilíbrio em Z e tabela β-PDF
import matplotlib.pyplot as plt
st = sc.st
sr = state_relation(PROPANE, sc.T_j, sc.T_inf, chi_loss=ch.F_s)   # perda radiativa = fração radiante
tab = beta_pdf_table(sr, st["Z_st"])
refs = sc.references(sr.T_ad_st)
print(f"{sr.source} · Z_st = {st['Z_st']:.4f} · T_ad = {sr.T_ad_st:.0f} K")
print(f"Delichatsios: Fr_f = {refs['delichatsios']['Fr_f']:.2f}, L = {refs['delichatsios']['L']:.1f} m · "
      f"Heskestad: L = {refs['L_heskestad']:.1f} m · Molina: X_rad = {refs['molina']['X_rad']:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(sr.Z, sr.T, lw=2); ax[0].axvline(st["Z_st"], ls="--", c="k", lw=1)
ax[0].set_xlim(0, 0.4); ax[0].set_xlabel("Z"); ax[0].set_ylabel("T [K]"); ax[0].set_title("Relação de estado")
for j in (0, 4, 8, 16):
    ax[1].plot(tab.Zt, tab.T[:, j], label=f"s = {tab.s[j]:.2f}")
ax[1].set_xlim(0, 0.4); ax[1].set_xlabel("Z̃"); ax[1].set_ylabel("T̃ [K]"); ax[1].legend()
ax[1].set_title("Média na β-PDF (variância de submalha)")
plt.tight_layout(); plt.show()

## 3 · LES com gráficos em tempo real
O painel abaixo é o mesmo quadro gravado no vídeo. Ele é atualizado a cada 0,1 s de tempo simulado.

In [ ]:
#@title 3 · LES 3D com painel ao vivo e gravação do vídeo
import torch
from flarekit.les import FlareLES, LESConfig
from flarekit.dashboard import run_live

PRESET = "gpu" if torch.cuda.is_available() else "cpu"   #@param ["gpu", "gpu_fino", "gpu_ultra", "cpu", "teste"] {allow-input: true}
T_END = 30.0       #@param {type:"number"}
T_AVG = 12.0       #@param {type:"number"}
FRAME_DT = 0.1     #@param {type:"number"}

cfg = LESConfig.preset(PRESET, H=sc.H, u_ref=sc.u_w, mdot=sc.mdot, u_jet=tip["u_j"], X_rad=ch.F_s,
                       T_inf=sc.T_inf, RH=sc.RH)
les = FlareLES(cfg, tab, sc.Q)
print(les.summary())
title = (f"Flare de propano · {sc.Q/1e6:.0f} MW · LES 3D · vento {sc.u_w:.1f} m/s" if sc.u_w > 0
         else f"Flare de propano · {sc.Q/1e6:.0f} MW · LES 3D · sem vento")
dash = run_live(les, sc, title, t_end=T_END, t_avg=T_AVG, frame_dt=FRAME_DT,
                video_path="flare_les.mp4", live=True, fps=20)
print(f"Concluído: {les.step_n} passos, {les.wall/60:.1f} min de solver, "
      f"{les.wall/les.step_n*1e3:.0f} ms/passo")

## 4 · Validação e segurança

In [ ]:
#@title 4 · Validação e segurança
from flarekit.report import summary_figure, print_validation, compose_video
from flarekit import safety
from IPython.display import Image, display

fig, rows, zones = summary_figure(les, sc, refs, "flare_resumo.png", title)
print_validation(rows)
print()
for z in zones:
    print(f"{z['nivel_kW_m2']:5.2f} kW/m² · área {z['area_m2']:7.0f} m² · alcance {z['raio_max_m']:5.0f} m · {z['descricao']}")
display(Image("flare_resumo.png"))

## 5 · Vídeo para o LinkedIn

In [ ]:
#@title 5 · Vídeo final (simulação + resumo) e download
from IPython.display import Video, display
compose_video("flare_les.mp4", "flare_resumo.png", "flare_linkedin.mp4", seconds=6, fps=20)
display(Video("flare_linkedin.mp4", embed=True, width=960))
try:
    from google.colab import files
    files.download("flare_linkedin.mp4")
except Exception:
    print("Arquivo salvo em flare_linkedin.mp4")

## 6 · Estudo de malha (opcional)

In [ ]:
#@title 6 · (Opcional) Estudo de malha: comprimento da chama vs. Δ
RUN_MESH_STUDY = False  #@param {type:"boolean"}
if RUN_MESH_STUDY:
    import matplotlib.pyplot as plt
    res = []
    for d in (1.0, 0.7, 0.5, 0.35):
        c = LESConfig.preset("gpu_fino", d_fine=d, H=sc.H, u_ref=sc.u_w, mdot=sc.mdot, u_jet=tip["u_j"], X_rad=ch.F_s)
        m = FlareLES(c, tab, sc.Q)
        while m.time < 20.0:
            m.step()
            if m.step_n % 20 == 0:
                m.diagnostics(m.time > 8.0)
        res.append((d, m.mean_flame_length(), m.mean_tilt()))
        print(f"Δ = {d:.2f} m → L = {res[-1][1]:.1f} m, inclinação = {res[-1][2]:.0f}°")
    d, L, a = map(np.array, zip(*res))
    plt.plot(d, L, "o-", label="LES"); plt.axhline(ch.L_b, ls="--", label="Chamberlain")
    plt.gca().invert_xaxis(); plt.xlabel("Δ fino [m]"); plt.ylabel("L [m]"); plt.legend(); plt.show()

## Notas de modelagem e limitações

- **Química rápida + equilíbrio** com fração de mistura; a perda radiativa entra como fração radiante constante (F_s de Chamberlain), como no FDS. Extinção local e eficiência de combustão (CE/DRE) exigiriam EDC/FPV.
- **Radiação no solo:** a emissão vem da zona luminosa (fuligem, relação de estado no lado rico) e é escalada para X_rad·Q, com transmissividade de Wayne em cada caminho. Não há solução da RTE com reabsorção dentro da chama.
- **Tip:** o jato real (0,27 m, 129 m/s) é sub-malha. A fonte injeta a vazão e o fluxo de quantidade de movimento reais num bloco de 2×2 células finas.
- **Numérica e malha:** o resultado é sensível à difusão numérica do transporte de Z. Com o limitador van Leer, a chama sob vento saía 35% mais curta; com Superbee (o do FDS) e Smagorinsky C_s = 0,1, Sc_t = 0,7, o comprimento converge para o de Chamberlain com o refino. Estudo de malha com vento de 8,9 m/s (Chamberlain: L = 27,4 m, α = 51°): Δ = 1,0 m → 10 m (van Leer); Δ = 0,7 m → 21,9 m; Δ = 0,5 m → 27,9 m e α = 56°. Sem vento, Δ = 0,7 m → 41,7 m (correlações: 47–54 m).
- **Validação:** comprimento e inclinação da chama e fluxo no solo comparados com Chamberlain (1987), o modelo industrial para flares sob vento, e com o API 521.

**Próximo passo (PINNeAPPle):** com algumas LES em velocidades de vento diferentes, treinar um surrogate (vento, vazão → mapa de radiação) com incerteza, para zonas de exclusão em tempo real.